# 04 — Embeddings: de One-Hot a Representaciones Contextuales

**Fase 2 — Representaciones** · LLM Lab (Etapa 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astorress/llm-lab/blob/main/notebooks/04_embeddings.ipynb)

## Objetivos de aprendizaje

Al terminar este notebook vas a poder:

1. **Explicar** por qué one-hot falla como representación de palabras y cómo la hipótesis distribucional motiva los embeddings densos.
2. **Derivar** la pérdida de skip-gram con negative sampling desde el softmax completo, incluyendo sus gradientes.
3. **Implementar** Word2Vec (skip-gram y CBOW) desde cero en PyTorch y entrenarlo sobre un corpus técnico bilingüe.
4. **Comparar** Word2Vec, GloVe y FastText: funciones objetivo, fortalezas y la conexión teórica con PMI (Levy & Goldberg).
5. **Medir** similitud semántica (coseno, euclidiana, producto punto) y resolver analogías con 3CosAdd.
6. **Distinguir** embeddings estáticos de contextuales, demostrando la polisemia con un modelo HuggingFace capa por capa.
7. **Construir** sentence embeddings (mean/CLS/max pooling), un mini buscador semántico y clustering por dominio.
8. **Visualizar** espacios de embeddings con PCA, t-SNE y UMAP, entendiendo qué se puede (y qué NO se puede) leer en cada mapa.

## Prerrequisitos

- Notebook 03 (tokenización) o conocimiento equivalente.
- PyTorch básico: `nn.Module`, `Dataset`, `DataLoader`, autograd.
- Álgebra lineal: producto punto, normas, eigenvectores (repasamos lo necesario).

## Tiempo estimado

| Perfil | Hardware | Tiempo |
|--------|----------|--------|
| SMALL  | Colab Free (T4) o CPU | ~35 min |
| LARGE  | Colab Pro (A100) | ~45 min |


## Tabla de contenidos

0. [Setup y corpus técnico bilingüe](#setup)
1. [Representaciones de palabras: one-hot, hipótesis distribucional y PPMI](#s1)
2. [Word2Vec desde cero: skip-gram con negative sampling](#s2)
3. [CBOW: la otra cara de Word2Vec](#s3)
4. [GloVe: factorización ponderada de co-ocurrencias](#s4)
5. [FastText: n-gramas de subpalabras](#s5)
6. [Similitud semántica: coseno, euclidiana y producto punto](#s6)
7. [Espacios vectoriales y aritmética de embeddings](#s7)
8. [Embeddings contextuales vs estáticos: el problema de la polisemia](#s8)
9. [Embeddings dentro del transformer](#s9)
10. [Sentence embeddings y buscador semántico](#s10)
11. [Clustering semántico](#s11)
12. [Visualización de espacios: PCA, t-SNE y UMAP](#s12)
13. [Resumen, ejercicios y referencias](#s13)


<a id="setup"></a>
## 0. Setup y corpus técnico bilingüe

Instalamos dependencias faltantes (sin reinstalar `torch` en Colab), detectamos hardware, fijamos seeds y construimos el corpus inline con el que entrenaremos **todos** los modelos desde cero.


In [ ]:
# ============================================================
# SETUP: dependencias, hardware, seeds
# ============================================================
import importlib.util
import subprocess
import sys

# Instalar solo lo que falte (no reinstalamos torch en Colab)
_DEPS = [
    ("gensim", "gensim"),
    ("umap-learn", "umap"),
    ("scikit-learn", "sklearn"),
    ("plotly", "plotly"),
    ("transformers", "transformers"),
]
for _pip_name, _module in _DEPS:
    if importlib.util.find_spec(_module) is None:
        print("Instalando {} ...".format(_pip_name))
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", _pip_name],
            check=False,
        )

import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
import seaborn as sns

# --- Detección de entorno ---
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    GPU_NAME = "Apple Silicon (MPS)"
    VRAM_GB = 0.0
else:
    DEVICE = torch.device("cpu")
    GPU_NAME = "CPU"
    VRAM_GB = 0.0

# --- Seeds para reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Estilo de gráficos ---
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12

print("Entorno: {}".format("Google Colab" if IN_COLAB else "local"))
print("Dispositivo: {} ({})".format(DEVICE, GPU_NAME))
if VRAM_GB > 0:
    print("VRAM: {:.1f} GB".format(VRAM_GB))
print("Seed: {}".format(SEED))


In [ ]:
# ============================================================
# PERFIL SMALL / LARGE
# ============================================================
# "auto": elige LARGE si hay GPU con >= 20 GB de VRAM (A100), si no SMALL.
# Forzable a mano: PROFILE = "SMALL" o PROFILE = "LARGE".
PROFILE = "auto"

CONFIG = {
    "SMALL": {
        "emb_dim": 64,        # dimensión de los embeddings propios
        "w2v_epochs": 120,    # epochs de skip-gram
        "cbow_epochs": 60,    # epochs de CBOW
        "glove_epochs": 250,  # epochs de GloVe simplificado
        "window": 3,          # ventana de contexto
        "neg_k": 5,           # negativos por par positivo
        "batch_size": 256,
        "lr": 2e-3,
    },
    "LARGE": {
        "emb_dim": 128,
        "w2v_epochs": 300,
        "cbow_epochs": 120,
        "glove_epochs": 500,
        "window": 4,
        "neg_k": 10,
        "batch_size": 512,
        "lr": 2e-3,
    },
}

if PROFILE == "auto":
    PROFILE = "LARGE" if VRAM_GB >= 20 else "SMALL"

CFG = CONFIG[PROFILE]
print("Perfil activo: {}".format(PROFILE))
for _k, _v in CFG.items():
    print("  {:<12} = {}".format(_k, _v))


### El corpus técnico bilingüe inline

Para entrenar Word2Vec, GloVe y FastText **desde cero** necesitamos un corpus pequeño pero denso en vocabulario técnico. Usamos ~130 oraciones que cubren los **21 dominios** del corpus del LLM Lab (ver `docs/00_vision_general.md`), mezclando español (~75%) e inglés (~25%).

Dos decisiones de diseño importantes:

- **Vocabulario recurrente:** términos como *transformador, voltaje, corriente, potencia, subestación, gradiente, red neuronal* aparecen muchas veces en contextos distintos — sin repetición, ningún método distribucional aprende nada.
- **Etiqueta de dominio por oración:** la usaremos en las secciones 11 y 12 para verificar si los clusters y las proyecciones 2D/3D recuperan la estructura temática.

> ⚠️ Un corpus de ~130 oraciones es *minúsculo* (Word2Vec original usó ~100B de tokens). Los embeddings resultantes serán ruidosos pero suficientes para **ver la mecánica** de cada algoritmo. Donde el corpus no alcance, recurrimos a modelos preentrenados con `try/except`.


In [ ]:
# ============================================================
# CORPUS: ~130 oraciones, 21 dominios, bilingüe
# Cada entrada: (dominio, oración)
# ============================================================
CORPUS = [
    # --- Machine Learning ---
    ("Machine Learning", "el modelo de machine learning aprende patrones a partir de los datos de entrenamiento"),
    ("Machine Learning", "el gradiente descendente minimiza la funcion de perdida ajustando los pesos del modelo"),
    ("Machine Learning", "la regularizacion evita el sobreajuste del modelo a los datos de entrenamiento"),
    ("Machine Learning", "the learning rate controls the size of each gradient descent step"),
    ("Machine Learning", "la validacion cruzada estima el error de generalizacion del modelo"),
    ("Machine Learning", "el modelo de regresion predice la demanda de energia electrica con los datos historicos"),
    # --- Deep Learning ---
    ("Deep Learning", "la red neuronal profunda apila capas de neuronas con funciones de activacion no lineales"),
    ("Deep Learning", "el algoritmo de backpropagation propaga el gradiente de la perdida hacia cada capa de la red"),
    ("Deep Learning", "la arquitectura transformer usa atencion para procesar secuencias de tokens en paralelo"),
    ("Deep Learning", "the transformer model replaced recurrent networks in language processing tasks"),
    ("Deep Learning", "la red neuronal convolucional extrae caracteristicas locales de la imagen"),
    ("Deep Learning", "el gradiente se desvanece cuando la red neuronal tiene demasiadas capas sin conexiones residuales"),
    # --- Ciencia de Datos ---
    ("Ciencia de Datos", "el cientifico de datos limpia y transforma los datos antes de entrenar el modelo"),
    ("Ciencia de Datos", "el analisis exploratorio revela la distribucion y las correlaciones de las variables"),
    ("Ciencia de Datos", "data quality issues degrade the performance of every downstream model"),
    ("Ciencia de Datos", "el pipeline de datos automatiza la ingesta la limpieza y la agregacion"),
    ("Ciencia de Datos", "la visualizacion de datos comunica patrones que las tablas ocultan"),
    ("Ciencia de Datos", "las series de tiempo de consumo electrico requieren features de calendario y clima"),
    # --- Inteligencia Artificial ---
    ("Inteligencia Artificial", "la inteligencia artificial generativa produce texto imagenes y codigo"),
    ("Inteligencia Artificial", "el agente de inteligencia artificial planifica acciones para alcanzar su objetivo"),
    ("Inteligencia Artificial", "large language models predict the next token from the previous context"),
    ("Inteligencia Artificial", "el modelo de lenguaje aprende representaciones del texto durante el preentrenamiento"),
    ("Inteligencia Artificial", "la busqueda heuristica explora el espacio de estados del problema"),
    ("Inteligencia Artificial", "el sistema experto codifica reglas del dominio electrico para diagnosticar fallas"),
    # --- Ingeniería Eléctrica ---
    ("Ingenieria Electrica", "el transformador eleva el voltaje para transmitir la potencia con menos perdidas"),
    ("Ingenieria Electrica", "el voltaje impulsa la corriente a traves del circuito electrico"),
    ("Ingenieria Electrica", "la presion impulsa el caudal a traves de la tuberia hidraulica"),
    ("Ingenieria Electrica", "la corriente alterna invierte su polaridad sesenta veces por segundo"),
    ("Ingenieria Electrica", "el motor electrico convierte la potencia electrica en potencia mecanica"),
    ("Ingenieria Electrica", "the transformer core concentrates the magnetic flux between both windings"),
    ("Ingenieria Electrica", "la ley de ohm relaciona el voltaje la corriente y la resistencia del circuito"),
    ("Ingenieria Electrica", "el caudal en la tuberia es analogo a la corriente en el circuito y la presion al voltaje"),
    # --- Sistemas Eléctricos de Potencia ---
    ("Sistemas de Potencia", "la subestacion transforma el voltaje de transmision al voltaje de distribucion"),
    ("Sistemas de Potencia", "el transformador de potencia de la subestacion se refrigera con aceite mineral"),
    ("Sistemas de Potencia", "el despacho economico asigna la generacion minimizando el costo del sistema"),
    ("Sistemas de Potencia", "la demanda de energia electrica alcanza su pico en las horas de la noche"),
    ("Sistemas de Potencia", "power flow studies compute voltage and current in every bus of the grid"),
    ("Sistemas de Potencia", "el rele de proteccion despeja la falla abriendo el interruptor de la subestacion"),
    ("Sistemas de Potencia", "la red electrica balancea la generacion y la demanda de potencia en tiempo real"),
    # --- Matemáticas ---
    ("Matematicas", "el eigenvector de la matriz no cambia de direccion bajo la transformacion lineal"),
    ("Matematicas", "la derivada mide la tasa de cambio instantanea de la funcion"),
    ("Matematicas", "el producto punto de dos vectores ortogonales es cero"),
    ("Matematicas", "the gradient vector points in the direction of steepest ascent of the function"),
    ("Matematicas", "la matriz de covarianza captura la variacion conjunta de las variables"),
    ("Matematicas", "el espacio vectorial cumple los axiomas de suma y multiplicacion por escalar"),
    # --- Estadística ---
    ("Estadistica", "la distribucion normal queda definida por su media y su varianza"),
    ("Estadistica", "el intervalo de confianza cuantifica la incertidumbre de la estimacion"),
    ("Estadistica", "la correlacion entre dos variables no implica causalidad"),
    ("Estadistica", "bayesian inference updates the prior distribution with the observed data"),
    ("Estadistica", "el test de hipotesis controla la probabilidad de un falso positivo"),
    ("Estadistica", "la varianza de la demanda electrica aumenta durante los eventos climaticos extremos"),
    # --- Optimización ---
    ("Optimizacion", "el optimizador adam adapta la tasa de aprendizaje con momentos del gradiente"),
    ("Optimizacion", "la funcion convexa garantiza que el minimo local sea el minimo global"),
    ("Optimizacion", "el problema de optimizacion minimiza la funcion objetivo sujeta a restricciones"),
    ("Optimizacion", "stochastic gradient descent uses noisy gradient estimates from mini batches"),
    ("Optimizacion", "el multiplicador de lagrange incorpora la restriccion en la funcion objetivo"),
    ("Optimizacion", "el despacho de generacion es un problema de optimizacion con restricciones de red"),
    # --- Programación ---
    ("Programacion", "la funcion recursiva se llama a si misma hasta alcanzar el caso base"),
    ("Programacion", "el codigo limpio usa nombres descriptivos y funciones cortas"),
    ("Programacion", "el test unitario verifica el comportamiento de cada funcion del modulo"),
    ("Programacion", "version control tracks every change in the source code of the project"),
    ("Programacion", "la estructura de datos correcta reduce la complejidad del algoritmo"),
    ("Programacion", "el patron de diseno separa la logica del negocio de la infraestructura"),
    # --- Python ---
    ("Python", "python es el lenguaje dominante para machine learning y ciencia de datos"),
    ("Python", "el decorador de python envuelve la funcion para extender su comportamiento"),
    ("Python", "la lista por comprension construye la coleccion en una sola linea de codigo"),
    ("Python", "numpy vectorizes numerical operations over entire arrays without explicit loops"),
    ("Python", "el entorno virtual aisla las dependencias de cada proyecto de python"),
    ("Python", "pandas agrupa y agrega los datos tabulares con pocas lineas de codigo"),
    # --- PostgreSQL ---
    ("PostgreSQL", "la base de datos postgresql garantiza transacciones con propiedades acid"),
    ("PostgreSQL", "el indice acelera la consulta evitando el escaneo completo de la tabla"),
    ("PostgreSQL", "la consulta sql une las tablas de clientes y consumos de energia"),
    ("PostgreSQL", "the query planner chooses the execution strategy with the lowest estimated cost"),
    ("PostgreSQL", "la vista materializada precalcula la agregacion para acelerar el reporte"),
    ("PostgreSQL", "el esquema relacional normaliza los datos para evitar redundancia"),
    # --- MongoDB ---
    ("MongoDB", "mongodb almacena documentos json flexibles sin esquema fijo"),
    ("MongoDB", "la coleccion de mongodb agrupa documentos con estructura similar"),
    ("MongoDB", "el pipeline de agregacion transforma los documentos en etapas sucesivas"),
    ("MongoDB", "sharding distributes the documents of a collection across multiple nodes"),
    ("MongoDB", "el indice compuesto de mongodb acelera las consultas con varios campos"),
    ("MongoDB", "la base de datos de documentos guarda las lecturas de los medidores electricos"),
    # --- AWS ---
    ("AWS", "aws ofrece computo almacenamiento y bases de datos como servicios en la nube"),
    ("AWS", "el bucket de s3 almacena los archivos del data lake con durabilidad alta"),
    ("AWS", "la instancia de ec2 escala verticalmente cambiando el tipo de maquina"),
    ("AWS", "iam policies control which user can access each cloud resource"),
    ("AWS", "la cola de mensajes desacopla los servicios distribuidos en la nube"),
    ("AWS", "el servicio de aws entrena el modelo de machine learning con gpu bajo demanda"),
    # --- Serverless Framework ---
    ("Serverless", "la funcion lambda ejecuta el codigo sin administrar servidores"),
    ("Serverless", "el framework serverless despliega la funcion y sus permisos con un solo comando"),
    ("Serverless", "el evento de s3 dispara la funcion lambda que procesa el archivo"),
    ("Serverless", "cold starts add latency to the first invocation of a serverless function"),
    ("Serverless", "la arquitectura serverless cobra solo por el tiempo de ejecucion consumido"),
    ("Serverless", "la funcion serverless ingiere las medidas de demanda electrica cada cinco minutos"),
    # --- Infraestructura como Código ---
    ("IaC", "la infraestructura como codigo declara los recursos de la nube en archivos versionados"),
    ("IaC", "terraform aplica el plan y converge la infraestructura al estado deseado"),
    ("IaC", "el modulo de terraform reutiliza la definicion de la red en varios ambientes"),
    ("IaC", "drift detection compares the declared state with the real infrastructure"),
    ("IaC", "la plantilla de cloudformation crea la base de datos y sus permisos"),
    ("IaC", "el codigo de infraestructura se revisa y prueba igual que el codigo de aplicacion"),
    # --- APIs ---
    ("APIs", "la api rest expone los recursos mediante verbos http y rutas"),
    ("APIs", "el endpoint de la api devuelve la prediccion de demanda en formato json"),
    ("APIs", "la autenticacion con token protege cada peticion a la api"),
    ("APIs", "rate limiting protects the api from abusive clients"),
    ("APIs", "el contrato de la api se documenta con un esquema openapi"),
    ("APIs", "la version de la api evoluciona sin romper a los clientes existentes"),
    # --- Arquitectura de Software ---
    ("Arquitectura", "la arquitectura de microservicios divide el sistema en servicios independientes"),
    ("Arquitectura", "el acoplamiento bajo y la cohesion alta facilitan el cambio del sistema"),
    ("Arquitectura", "la capa de dominio no depende de la base de datos ni del framework"),
    ("Arquitectura", "event driven architecture reacts to messages instead of direct calls"),
    ("Arquitectura", "el diagrama de arquitectura muestra los servicios y sus dependencias"),
    ("Arquitectura", "la decision de arquitectura se documenta con su contexto y consecuencias"),
    # --- DevOps ---
    ("DevOps", "el pipeline de integracion continua compila prueba y despliega el codigo"),
    ("DevOps", "el contenedor de docker empaqueta la aplicacion con sus dependencias"),
    ("DevOps", "kubernetes orquesta los contenedores y reinicia los que fallan"),
    ("DevOps", "observability combines logs metrics and traces to debug production systems"),
    ("DevOps", "el despliegue canario expone la version nueva a una fraccion del trafico"),
    ("DevOps", "la metrica de produccion alerta cuando la latencia del servicio se degrada"),
    # --- Data Engineering ---
    ("Data Engineering", "el pipeline etl extrae transforma y carga los datos al data warehouse"),
    ("Data Engineering", "la tabla particionada por fecha acelera las consultas del data lake"),
    ("Data Engineering", "el orquestador programa las tareas del pipeline con dependencias"),
    ("Data Engineering", "streaming pipelines process events with latency of seconds"),
    ("Data Engineering", "la calidad de datos se valida con esquemas y pruebas automaticas"),
    ("Data Engineering", "el pipeline de datos agrega las lecturas de los medidores de energia por hora"),
    # --- Sistemas Distribuidos ---
    ("Sistemas Distribuidos", "el sistema distribuido replica los datos para tolerar fallas de nodos"),
    ("Sistemas Distribuidos", "el consenso distribuido acuerda un valor entre nodos que pueden fallar"),
    ("Sistemas Distribuidos", "el teorema cap obliga a elegir entre consistencia y disponibilidad ante particiones"),
    ("Sistemas Distribuidos", "the load balancer distributes requests across the replicas of the service"),
    ("Sistemas Distribuidos", "el reloj logico ordena los eventos sin un reloj global compartido"),
    ("Sistemas Distribuidos", "la red de la subestacion envia las medidas a un sistema distribuido de monitoreo"),
]

# Grupos macro de dominio (para clustering y colores en visualizaciones)
DOMAIN_GROUP = {
    "Machine Learning": "ML/IA", "Deep Learning": "ML/IA",
    "Ciencia de Datos": "ML/IA", "Inteligencia Artificial": "ML/IA",
    "Ingenieria Electrica": "Electrica", "Sistemas de Potencia": "Electrica",
    "Matematicas": "Matematicas", "Estadistica": "Matematicas", "Optimizacion": "Matematicas",
    "Programacion": "Software", "Python": "Software", "APIs": "Software", "Arquitectura": "Software",
    "PostgreSQL": "Datos/Infra", "MongoDB": "Datos/Infra", "AWS": "Datos/Infra",
    "Serverless": "Datos/Infra", "IaC": "Datos/Infra", "DevOps": "Datos/Infra",
    "Data Engineering": "Datos/Infra", "Sistemas Distribuidos": "Datos/Infra",
}

SENTENCES = [s for _, s in CORPUS]
DOMAINS = [d for d, _ in CORPUS]
GROUPS = [DOMAIN_GROUP[d] for d in DOMAINS]

print("Oraciones: {}".format(len(CORPUS)))
print("Dominios: {}".format(len(set(DOMAINS))))
print("Grupos macro: {}".format(sorted(set(GROUPS))))


In [ ]:
# ============================================================
# Tokenización simple por palabras + vocabulario
# ============================================================
def tokenize(text):
    """Tokenizador minimalista: minúsculas + solo letras (incluye ñ/acentos)."""
    return re.findall(r"[a-záéíóúüñ]+", text.lower())

TOKENIZED = [tokenize(s) for s in SENTENCES]  # lista de listas de tokens
ALL_TOKENS = [t for sent in TOKENIZED for t in sent]

word_counts = Counter(ALL_TOKENS)
MIN_COUNT = 2  # descartamos hapax legomena (aparecen 1 vez): no se puede aprender de ellas

vocab = sorted([w for w, c in word_counts.items() if c >= MIN_COUNT])
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

print("Tokens totales: {}".format(len(ALL_TOKENS)))
print("Vocabulario completo: {} tipos".format(len(word_counts)))
print("Vocabulario con count >= {}: {} tipos".format(MIN_COUNT, V))
print()
print("Top 15 palabras:")
for w, c in word_counts.most_common(15):
    print("  {:<15} {}".format(w, c))

# Términos técnicos que seguiremos a lo largo del notebook
TECH_TERMS = ["transformador", "voltaje", "corriente", "potencia", "subestacion",
              "gradiente", "red", "neuronal", "modelo", "datos", "api", "pipeline"]
for t in TECH_TERMS:
    assert t in word2idx, "Falta el término técnico: " + t
print()
print("Todos los términos técnicos de seguimiento están en el vocabulario ✓")


<a id="s1"></a>
## 1. Representaciones de palabras: one-hot y sus problemas

### 1.1 One-hot encoding

La forma más simple de representar una palabra como vector: un vector de dimensión $|V|$ (tamaño del vocabulario) con un 1 en la posición de la palabra y 0 en el resto:

$$\mathbf{e}_{\text{voltaje}} = [0, 0, \dots, \underbrace{1}_{\text{posición } i}, \dots, 0] \in \{0,1\}^{|V|}$$

**Problema 1 — Ortogonalidad total.** El producto punto entre dos one-hot distintos es siempre 0:

$$\mathbf{e}_i^\top \mathbf{e}_j = \sum_k e_{ik} \, e_{jk} = 0 \quad \forall\, i \neq j$$

Para el one-hot, *voltaje* está exactamente igual de lejos de *corriente* que de *mongodb*. **Toda la semántica se pierde**: no hay noción de similitud.

**Problema 2 — Dimensión.** Con un vocabulario realista de $|V| = 50{,}000$, cada palabra es un vector de 50.000 dimensiones con un solo bit de información. La matriz de una capa que reciba este vector tendría $50{,}000 \times d$ parámetros solo de entrada — y casi todos multiplicarían ceros.

**Problema 3 — Sin generalización.** Lo aprendido sobre *transformador* no transfiere nada a *transformadores*: son dimensiones independientes.

### 1.2 La hipótesis distribucional

> *"You shall know a word by the company it keeps"* — J.R. Firth, 1957.

Palabras que aparecen en **contextos similares** tienen **significados similares**. *Voltaje* y *corriente* comparten vecinos (*circuito*, *transformador*, *ley de ohm*...), mientras que *mongodb* aparece junto a *documentos*, *coleccion*, *indice*. Todos los métodos de este notebook — PPMI, Word2Vec, GloVe, FastText, BERT — son formas distintas de **convertir estadísticas de co-ocurrencia en geometría**: contextos compartidos → vectores cercanos.


In [ ]:
# ============================================================
# One-hot: ortogonalidad y dimensión
# ============================================================
def one_hot(word):
    v = np.zeros(V, dtype=np.float32)  # (V,)
    v[word2idx[word]] = 1.0
    return v

e_voltaje = one_hot("voltaje")      # (V,)
e_corriente = one_hot("corriente")  # (V,)
e_mongodb = one_hot("mongodb")      # (V,)

def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

print("Dimensión de cada vector one-hot: {}".format(V))
print()
print("Producto punto voltaje·corriente : {:.1f}".format(np.dot(e_voltaje, e_corriente)))
print("Producto punto voltaje·mongodb   : {:.1f}".format(np.dot(e_voltaje, e_mongodb)))
print()
print("Coseno(voltaje, corriente) = {:.3f}  <- ¡deberían parecerse!".format(cosine(e_voltaje, e_corriente)))
print("Coseno(voltaje, mongodb)   = {:.3f}  <- igual de 'lejos'".format(cosine(e_voltaje, e_mongodb)))

# Verificación: TODOS los pares distintos son ortogonales
assert np.dot(e_voltaje, e_corriente) == 0.0
assert cosine(e_voltaje, e_voltaje) > 0.999
print()
print("One-hot no codifica NINGUNA noción de similitud semántica ✓ (verificado)")


### 1.3 Matriz de co-ocurrencia y PPMI

**Matriz de co-ocurrencia** $X \in \mathbb{R}^{|V| \times |V|}$: la entrada $X_{wc}$ cuenta cuántas veces la palabra $c$ aparece dentro de una ventana de $\pm L$ posiciones alrededor de $w$. Cada fila de $X$ ya es un "embedding" disperso: el perfil distribucional de la palabra.

**Problema de las cuentas crudas:** palabras frecuentes (*de*, *la*, *el*) co-ocurren con todo, dominando la matriz sin aportar semántica. Necesitamos medir si dos palabras co-ocurren **más de lo que se esperaría por azar**.

### Derivación de PMI

Si $w$ y $c$ fueran estadísticamente independientes: $P(w, c) = P(w)\,P(c)$. El *Pointwise Mutual Information* mide la desviación respecto a esa independencia (en escala logarítmica):

$$\text{PMI}(w, c) = \log \frac{P(w, c)}{P(w)\,P(c)}$$

- $\text{PMI} > 0$: co-ocurren **más** de lo esperado por azar → asociación real (*voltaje*–*corriente*).
- $\text{PMI} = 0$: independientes.
- $\text{PMI} < 0$: co-ocurren menos de lo esperado — pero estimar esto con un corpus finito es poco fiable ($P(w,c)$ muy pequeñas → $\log \to -\infty$ para pares jamás vistos).

Por eso se usa el **PMI positivo**:

$$\text{PPMI}(w, c) = \max\big(0,\; \text{PMI}(w, c)\big)$$

Con estimaciones de máxima verosimilitud a partir de la matriz $X$ (donde $N = \sum_{w,c} X_{wc}$):

$$\hat P(w,c) = \frac{X_{wc}}{N}, \qquad \hat P(w) = \frac{\sum_c X_{wc}}{N}, \qquad \hat P(c) = \frac{\sum_w X_{wc}}{N}$$

**Ejemplo numérico verificable a mano.** Supongamos $N = 100$ pares de co-ocurrencia en total, con $X_{wc} = 4$, $\sum_c X_{wc} = 10$ (la palabra $w$ participa en 10 pares) y $\sum_w X_{wc} = 8$:

$$\text{PMI} = \log \frac{4/100}{(10/100)(8/100)} = \log \frac{0.04}{0.008} = \log 5 \approx 1.609$$

$w$ y $c$ co-ocurren **5× más** de lo que predeciría el azar. Verificaremos este número en código.


In [ ]:
# ============================================================
# Matriz de co-ocurrencia (ventana simétrica)
# ============================================================
def build_cooccurrence(tokenized_sents, word2idx, window):
    """Cuenta co-ocurrencias palabra-contexto en ventana de +-window."""
    V = len(word2idx)
    X = np.zeros((V, V), dtype=np.float64)  # (V, V)
    for sent in tokenized_sents:
        ids = [word2idx[t] for t in sent if t in word2idx]
        for i, w in enumerate(ids):
            lo = max(0, i - window)
            hi = min(len(ids), i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    X[w, ids[j]] += 1.0
    return X

X_cooc = build_cooccurrence(TOKENIZED, word2idx, window=CFG["window"])  # (V, V)
N_pairs = X_cooc.sum()

print("Matriz de co-ocurrencia: shape {}".format(X_cooc.shape))
print("Pares de co-ocurrencia totales: {:.0f}".format(N_pairs))
print("Densidad: {:.2%} de entradas no nulas".format((X_cooc > 0).mean()))
print()

# ¿Con qué co-ocurre 'transformador'?
row = X_cooc[word2idx["transformador"]]
top = np.argsort(-row)[:8]
print("Co-ocurrencias más frecuentes de 'transformador':")
for i in top:
    if row[i] > 0:
        print("  {:<15} {:.0f}".format(idx2word[i], row[i]))


In [ ]:
# ============================================================
# PPMI: de cuentas crudas a asociación estadística
# ============================================================
def ppmi_matrix(X):
    """PPMI(w,c) = max(0, log[ P(w,c) / (P(w) P(c)) ])."""
    N = X.sum()
    P_wc = X / N                      # (V, V)  P(w,c)
    P_w = X.sum(axis=1, keepdims=True) / N   # (V, 1)  P(w)
    P_c = X.sum(axis=0, keepdims=True) / N   # (1, V)  P(c)
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log(P_wc / (P_w * P_c))     # (V, V)
    pmi[~np.isfinite(pmi)] = 0.0
    return np.maximum(pmi, 0.0)

# --- Verificación del ejemplo numérico de la celda anterior ---
X_toy = np.array([[4.0, 6.0],
                  [4.0, 86.0]])  # N=100, fila 0 suma 10, columna 0 suma 8
pmi_toy = np.log((X_toy[0, 0] / 100) / ((10 / 100) * (8 / 100)))
assert abs(pmi_toy - math.log(5)) < 1e-12, "El ejemplo manual no cuadra"
print("Ejemplo manual: PMI = log(5) = {:.4f} ✓".format(pmi_toy))

X_ppmi = ppmi_matrix(X_cooc)  # (V, V)
print()
print("PPMI: {:.2%} de entradas positivas".format((X_ppmi > 0).mean()))
print()

# Comparación: vecinos por cuentas crudas vs por PPMI
w = word2idx["voltaje"]
print("Asociaciones más fuertes de 'voltaje' según PPMI:")
for i in np.argsort(-X_ppmi[w])[:8]:
    if X_ppmi[w, i] > 0:
        print("  {:<15} PPMI={:.2f}  (cuenta cruda={:.0f})".format(idx2word[i], X_ppmi[w, i], X_cooc[w, i]))
print()
print("Nota: palabras funcionales frecuentes ('de','la','el') pierden peso con PPMI,")
print("porque co-ocurren con TODO y eso es exactamente lo esperado por azar.")


In [ ]:
# ============================================================
# Visualización: heatmap de la submatriz PPMI de términos técnicos
# ============================================================
heat_terms = ["transformador", "voltaje", "corriente", "potencia", "subestacion",
              "circuito", "red", "neuronal", "gradiente", "modelo", "datos",
              "pipeline", "api", "funcion", "base"]
heat_terms = [t for t in heat_terms if t in word2idx]
idxs = [word2idx[t] for t in heat_terms]
sub = X_ppmi[np.ix_(idxs, idxs)]  # (k, k)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(X_cooc[np.ix_(idxs, idxs)], ax=axes[0], cmap="Blues",
            xticklabels=heat_terms, yticklabels=heat_terms,
            annot=True, fmt=".0f", annot_kws={"size": 7}, cbar_kws={"label": "cuentas"})
axes[0].set_title("Co-ocurrencias crudas")
sns.heatmap(sub, ax=axes[1], cmap="Reds",
            xticklabels=heat_terms, yticklabels=heat_terms,
            annot=True, fmt=".1f", annot_kws={"size": 7}, cbar_kws={"label": "PPMI"})
axes[1].set_title("PPMI (asociación más allá del azar)")
for ax in axes:
    ax.tick_params(axis="x", rotation=60)
plt.suptitle("La estructura de bloques temáticos emerge con PPMI", y=1.02)
plt.tight_layout()
plt.show()

print("Observa el bloque eléctrico (transformador/voltaje/corriente/potencia/subestacion)")
print("y el bloque ML (red/neuronal/gradiente/modelo): la hipótesis distribucional en acción.")


<a id="s2"></a>
## 2. Word2Vec desde cero: skip-gram con negative sampling

PPMI da vectores **dispersos** de dimensión $|V|$. Word2Vec (Mikolov et al., 2013) aprende vectores **densos** de dimensión $d \ll |V|$ resolviendo una tarea de predicción: en la variante **skip-gram**, dada la palabra central, predecir sus palabras de contexto.

### 2.1 Arquitectura skip-gram

```mermaid
flowchart LR
    A["palabra central w<br/>(índice)"] --> B["W_in: lookup<br/>(V, d)"]
    B --> C["v_w ∈ R^d"]
    C --> D["puntaje u_c · v_w<br/>contra W_out (V, d)"]
    D --> E["P(contexto c | w)<br/>softmax sobre V"]
```

*(Descripción textual: la palabra central se convierte en un vector denso $\mathbf{v}_w$ vía la matriz de entrada $W_{in}$; el puntaje con cada posible palabra de contexto es el producto punto con su vector en la matriz de salida $W_{out}$; el softmax sobre el vocabulario da la probabilidad.)*

Dos matrices de parámetros:

- $W_{in} \in \mathbb{R}^{|V| \times d}$ — vectores de **palabra central** $\mathbf{v}_w$ (los "embeddings" que conservamos al final).
- $W_{out} \in \mathbb{R}^{|V| \times d}$ — vectores de **contexto** $\mathbf{u}_c$.

Para cada ocurrencia de $w$ con $c$ en su ventana, queremos maximizar:

$$P(c \mid w) = \frac{\exp(\mathbf{u}_c^\top \mathbf{v}_w)}{\sum_{c'=1}^{|V|} \exp(\mathbf{u}_{c'}^\top \mathbf{v}_w)}$$


### 2.2 Del softmax completo a negative sampling

#### Por qué el softmax completo es intratable

La log-verosimilitud negativa de un par $(w, c)$:

$$\mathcal{L} = -\log P(c \mid w) = -\mathbf{u}_c^\top \mathbf{v}_w + \log \sum_{c'=1}^{|V|} \exp(\mathbf{u}_{c'}^\top \mathbf{v}_w)$$

Su gradiente respecto a $\mathbf{v}_w$:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_w} = -\mathbf{u}_c + \sum_{c'=1}^{|V|} \underbrace{P(c' \mid w)}_{\text{softmax}} \, \mathbf{u}_{c'} = -\mathbf{u}_c + \mathbb{E}_{c' \sim P(\cdot \mid w)}[\mathbf{u}_{c'}]$$

El término de normalización exige **recorrer todo el vocabulario en cada paso**: con $|V| = 10^5$–$10^6$ y miles de millones de pares de entrenamiento, es computacionalmente prohibitivo ($O(|V| \cdot d)$ por par).

#### Negative sampling: convertirlo en clasificación binaria

Idea (derivada de *Noise Contrastive Estimation*, NCE): en lugar de "¿cuál de las $|V|$ palabras es el contexto?", responder **"¿este par $(w,c)$ es real o ruido?"**. Para cada par positivo observado, muestreamos $K$ palabras "negativas" $n_1, \dots, n_K$ de una distribución de ruido $P_n$ y entrenamos un clasificador logístico:

$$P(\text{real} \mid w, c) = \sigma(\mathbf{u}_c^\top \mathbf{v}_w), \qquad \sigma(x) = \frac{1}{1 + e^{-x}}$$

La pérdida de **SGNS** (skip-gram with negative sampling) para un par positivo:

$$\mathcal{L}_{\text{SGNS}} = -\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) \;-\; \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$$

(usamos $\sigma(-x) = 1 - \sigma(x)$: queremos que los negativos tengan probabilidad baja de ser "reales").

#### Derivación del gradiente

Con $s_c = \mathbf{u}_c^\top \mathbf{v}_w$ y usando $\frac{d}{dx}\log\sigma(x) = 1 - \sigma(x) = \sigma(-x)$:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_w} = \big(\sigma(s_c) - 1\big)\,\mathbf{u}_c + \sum_{k=1}^{K} \sigma(s_{n_k})\,\mathbf{u}_{n_k}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{u}_c} = \big(\sigma(s_c) - 1\big)\,\mathbf{v}_w, \qquad \frac{\partial \mathcal{L}}{\partial \mathbf{u}_{n_k}} = \sigma(s_{n_k})\,\mathbf{v}_w$$

**Lectura intuitiva:** si el modelo ya clasifica bien ($\sigma(s_c) \to 1$ para el positivo, $\sigma(s_{n_k}) \to 0$ para los negativos), los gradientes se anulan. Si no, $\mathbf{v}_w$ se **acerca** a $\mathbf{u}_c$ y se **aleja** de los $\mathbf{u}_{n_k}$. El costo por par baja de $O(|V| d)$ a $O(K d)$ con $K \approx 5$–$20$.

> **NCE vs negative sampling:** NCE es un estimador consistente de la verosimilitud del softmax (incluye el término $\log(K \cdot P_n(c))$ en el clasificador). Negative sampling lo *simplifica* eliminando ese término: ya no estima el softmax, pero funciona mejor en la práctica para aprender representaciones. Levy & Goldberg (2014) demostraron qué optimiza realmente — lo veremos en la sección 4.


### 2.3 Distribución de negativos y subsampling

#### La distribución $f^{3/4}$

¿De dónde muestrear los negativos? Mikolov et al. encontraron empíricamente que ni la distribución unigram $f(w)$ (sobre-muestrea palabras funcionales) ni la uniforme (sobre-muestrea palabras rarísimas) funcionan bien. El punto dulce:

$$P_n(w) = \frac{f(w)^{3/4}}{\sum_{w'} f(w')^{3/4}}$$

El exponente $0.75 < 1$ **comprime** la distribución: aplana las frecuencias altas y eleva las bajas. Ejemplo: si $f(\text{de}) = 0.05$ y $f(\text{subestacion}) = 0.001$, la razón pasa de $50\times$ a $(0.05/0.001)^{0.75} \approx 18.8\times$.

#### Subsampling de palabras frecuentes

Las palabras muy frecuentes (*de*, *la*, *el*) aportan poca información ("el par (*voltaje*, *de*) no dice casi nada") pero dominan los datos. Antes de generar pares, cada ocurrencia de $w$ se **descarta** con probabilidad:

$$P_{\text{descartar}}(w) = 1 - \sqrt{\frac{t}{f(w)}}$$

donde $t$ es un umbral ($10^{-5}$ en el paper para corpus gigantes; usaremos $10^{-2}$ porque nuestro corpus es diminuto y las frecuencias relativas son altas). Solo afecta a palabras con $f(w) > t$. Efecto doble: acelera el entrenamiento y **amplía la ventana efectiva** (al eliminar palabras funcionales, palabras de contenido lejanas quedan adyacentes).


In [ ]:
# ============================================================
# Frecuencias, subsampling y distribución de negativos f^0.75
# ============================================================
freqs = np.array([word_counts[idx2word[i]] for i in range(V)], dtype=np.float64)  # (V,)
rel_freqs = freqs / freqs.sum()  # f(w), (V,)

# --- Subsampling: probabilidad de DESCARTAR cada ocurrencia ---
T_SUB = 1e-2  # umbral alto: corpus diminuto
p_discard = np.maximum(0.0, 1.0 - np.sqrt(T_SUB / rel_freqs))  # (V,)

print("Probabilidad de descarte (subsampling, t={:.0e}):".format(T_SUB))
for w in ["de", "la", "el", "datos", "transformador", "subestacion"]:
    i = word2idx[w]
    print("  {:<15} f={:.4f}  P(descartar)={:.2f}".format(w, rel_freqs[i], p_discard[i]))

# --- Distribución de negativos: f^0.75 ---
noise_probs = rel_freqs ** 0.75
noise_probs = noise_probs / noise_probs.sum()  # (V,)
noise_probs_t = torch.tensor(noise_probs, dtype=torch.float32)  # para torch.multinomial

# Visualizar el efecto del exponente 0.75
order = np.argsort(-rel_freqs)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(rel_freqs[order], label="unigram f(w)", lw=2)
axes[0].plot(noise_probs[order], label="f(w)^0.75 normalizada", lw=2)
axes[0].set_xlabel("rango de la palabra"); axes[0].set_ylabel("probabilidad")
axes[0].set_title("f^0.75 aplana la distribución"); axes[0].legend(); axes[0].set_yscale("log")

ratio = noise_probs[order] / rel_freqs[order]
axes[1].plot(ratio, color="purple", lw=2)
axes[1].axhline(1.0, color="gray", ls="--")
axes[1].set_xlabel("rango de la palabra"); axes[1].set_ylabel("P_noise / P_unigram")
axes[1].set_title("Palabras raras se sobre-muestrean (ratio > 1),\nfrecuentes se sub-muestrean (ratio < 1)")
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Dataset de pares (central, contexto) con subsampling
# ============================================================
class SkipGramDataset(Dataset):
    """Genera pares (palabra_central, palabra_contexto) con ventana dinámica."""

    def __init__(self, tokenized_sents, word2idx, window, p_discard, seed=SEED):
        rng = np.random.default_rng(seed)
        self.pairs = []
        for sent in tokenized_sents:
            ids = [word2idx[t] for t in sent if t in word2idx]
            # subsampling: descartar ocurrencias de palabras muy frecuentes
            ids = [i for i in ids if rng.random() >= p_discard[i]]
            for pos, center in enumerate(ids):
                # ventana dinámica (como el word2vec original): tamaño ~ U[1, window]
                win = rng.integers(1, window + 1)
                lo = max(0, pos - win)
                hi = min(len(ids), pos + win + 1)
                for ctx_pos in range(lo, hi):
                    if ctx_pos != pos:
                        self.pairs.append((center, ids[ctx_pos]))
        self.pairs = np.array(self.pairs, dtype=np.int64)  # (N_pairs, 2)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        c, ctx = self.pairs[idx]
        return torch.tensor(c), torch.tensor(ctx)

sg_dataset = SkipGramDataset(TOKENIZED, word2idx, CFG["window"], p_discard)
sg_loader = DataLoader(sg_dataset, batch_size=CFG["batch_size"], shuffle=True, drop_last=False)

print("Pares de entrenamiento (tras subsampling): {}".format(len(sg_dataset)))
print("Batches por epoch: {}".format(len(sg_loader)))
print()
print("Ejemplos de pares (central -> contexto):")
for c, ctx in sg_dataset.pairs[:8]:
    print("  {:<15} -> {}".format(idx2word[c], idx2word[ctx]))


In [ ]:
# ============================================================
# Modelo SGNS: dos tablas de embeddings + pérdida con negativos
# ============================================================
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, emb_dim)   # W_in:  v_w, (V, d)
        self.out_emb = nn.Embedding(vocab_size, emb_dim)  # W_out: u_c, (V, d)
        # Inicialización pequeña (como word2vec.c): U(-0.5/d, 0.5/d)
        bound = 0.5 / emb_dim
        nn.init.uniform_(self.in_emb.weight, -bound, bound)
        nn.init.zeros_(self.out_emb.weight)

    def forward(self, center, context, negatives):
        # center: (B,)  context: (B,)  negatives: (B, K)
        v = self.in_emb(center)        # (B, d)
        u_pos = self.out_emb(context)  # (B, d)
        u_neg = self.out_emb(negatives)  # (B, K, d)

        s_pos = (v * u_pos).sum(dim=-1)               # (B,)    u_c · v_w
        s_neg = torch.bmm(u_neg, v.unsqueeze(-1)).squeeze(-1)  # (B, K) u_nk · v_w

        # L = -log σ(s_pos) - Σ log σ(-s_neg)   [derivada en la sección 2.2]
        loss_pos = -F.logsigmoid(s_pos)               # (B,)
        loss_neg = -F.logsigmoid(-s_neg).sum(dim=-1)  # (B,)
        return (loss_pos + loss_neg).mean()

sg_model = SkipGramNS(V, CFG["emb_dim"]).to(DEVICE)
n_params = sum(p.numel() for p in sg_model.parameters())
print(sg_model)
print()
print("Parámetros: {:,}  (= 2 x V x d = 2 x {} x {})".format(n_params, V, CFG["emb_dim"]))


In [ ]:
# ============================================================
# Entrenamiento de skip-gram con negative sampling
# ============================================================
optimizer = torch.optim.Adam(sg_model.parameters(), lr=CFG["lr"])
K = CFG["neg_k"]
sg_losses = []

sg_model.train()
for epoch in range(CFG["w2v_epochs"]):
    epoch_loss = 0.0
    for center, context in sg_loader:
        center = center.to(DEVICE)    # (B,)
        context = context.to(DEVICE)  # (B,)
        # Muestrear K negativos por par desde P_n(w) ∝ f^0.75
        negatives = torch.multinomial(
            noise_probs_t, num_samples=center.size(0) * K, replacement=True
        ).view(center.size(0), K).to(DEVICE)  # (B, K)

        loss = sg_model(center, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    sg_losses.append(epoch_loss / len(sg_loader))
    if (epoch + 1) % max(1, CFG["w2v_epochs"] // 10) == 0:
        print("epoch {:>4d}/{} | loss = {:.4f}".format(epoch + 1, CFG["w2v_epochs"], sg_losses[-1]))

# Los embeddings finales son W_in (práctica estándar; también se usa W_in + W_out)
W_sg = sg_model.in_emb.weight.detach().cpu().numpy()  # (V, d)
print()
print("Embeddings skip-gram entrenados: shape {}".format(W_sg.shape))


In [ ]:
# ============================================================
# Curva de pérdida
# ============================================================
plt.figure(figsize=(8, 4))
plt.plot(sg_losses, lw=2, color="steelblue")
plt.xlabel("epoch"); plt.ylabel("pérdida SGNS media")
plt.title("Skip-gram con negative sampling — curva de entrenamiento ({})".format(PROFILE))
plt.tight_layout(); plt.show()

# Cota de referencia: con scores=0, L = -log σ(0) * (1+K) = log(2) * (1+K)
ref = math.log(2) * (1 + K)
print("Pérdida inicial teórica (scores=0): log(2)x(1+K) = {:.3f}".format(ref))
print("Pérdida inicial observada: {:.3f}".format(sg_losses[0]))
print("Pérdida final: {:.3f}".format(sg_losses[-1]))
assert sg_losses[-1] < sg_losses[0], "La pérdida debería bajar"
print("La pérdida desciende de forma estable ✓")


<a id="s3"></a>
## 3. CBOW: la otra cara de Word2Vec

**CBOW** (*Continuous Bag of Words*) invierte la tarea de skip-gram: en lugar de predecir el contexto desde la palabra central, **predice la palabra central desde el promedio de su contexto**.

```mermaid
flowchart LR
    subgraph SkipGram["Skip-gram"]
        A1["central: 'voltaje'"] --> B1["predice c/u de:<br/>'el', 'impulsa', 'la', 'corriente'"]
    end
    subgraph CBOW["CBOW"]
        A2["contexto promediado:<br/>('el'+'impulsa'+'la'+'corriente')/4"] --> B2["predice: 'voltaje'"]
    end
```

*(Descripción textual: skip-gram genera 4 ejemplos de entrenamiento a partir de una ventana — la central predice cada vecino por separado; CBOW genera 1 solo ejemplo — el promedio de los 4 vecinos predice la central.)*

El vector de entrada es el **promedio** de los embeddings del contexto:

$$\bar{\mathbf{v}} = \frac{1}{2L} \sum_{-L \le j \le L,\, j \neq 0} \mathbf{v}_{w_{t+j}}, \qquad \mathcal{L} = -\log \sigma(\mathbf{u}_{w_t}^\top \bar{\mathbf{v}}) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^\top \bar{\mathbf{v}})$$

### ¿Cuándo conviene cada uno?

| | Skip-gram | CBOW |
|---|---|---|
| Ejemplos por ventana | $2L$ (uno por vecino) | $1$ (contexto promediado) |
| Velocidad | más lento | **más rápido** (~2-5×) |
| Palabras **raras** | **mejor** — cada ocurrencia genera muchos updates dedicados | peor — su señal se diluye en el promedio |
| Palabras frecuentes / sintaxis | bien | ligeramente mejor |
| Corpus pequeño | **recomendado** | necesita más datos |

Regla práctica de Mikolov: *skip-gram para corpus pequeños y vocabulario raro; CBOW para corpus enormes donde la velocidad importa*. El promediado de CBOW actúa como suavizado: bueno con mucha señal, destructivo con poca.


In [ ]:
# ============================================================
# CBOW desde cero (reutiliza negative sampling de la sección 2)
# ============================================================
class CBOWDataset(Dataset):
    """Ejemplos (contexto fijo de 2*window con padding, central)."""

    def __init__(self, tokenized_sents, word2idx, window):
        self.window = window
        self.examples = []
        for sent in tokenized_sents:
            ids = [word2idx[t] for t in sent if t in word2idx]
            for pos, center in enumerate(ids):
                ctx = ids[max(0, pos - window):pos] + ids[pos + 1:pos + window + 1]
                if ctx:
                    self.examples.append((ctx, center))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ctx, center = self.examples[idx]
        # padding con -1 hasta 2*window (lo enmascaramos en el modelo)
        padded = ctx + [-1] * (2 * self.window - len(ctx))
        return torch.tensor(padded), torch.tensor(center)

class CBOW_NS(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=vocab_size)  # +1: slot de padding
        self.out_emb = nn.Embedding(vocab_size, emb_dim)
        bound = 0.5 / emb_dim
        nn.init.uniform_(self.in_emb.weight, -bound, bound)
        with torch.no_grad():
            self.in_emb.weight[vocab_size].zero_()
        nn.init.zeros_(self.out_emb.weight)
        self.pad_idx = vocab_size

    def forward(self, context, center, negatives):
        # context: (B, 2L) con -1 de padding | center: (B,) | negatives: (B, K)
        mask = (context >= 0)                          # (B, 2L)
        ctx_safe = torch.where(mask, context, torch.full_like(context, self.pad_idx))
        emb = self.in_emb(ctx_safe)                    # (B, 2L, d) — padding da vector 0
        n_real = mask.sum(dim=1, keepdim=True).clamp(min=1)  # (B, 1)
        v_bar = emb.sum(dim=1) / n_real                # (B, d)  promedio del contexto

        s_pos = (v_bar * self.out_emb(center)).sum(-1)                    # (B,)
        s_neg = torch.bmm(self.out_emb(negatives), v_bar.unsqueeze(-1)).squeeze(-1)  # (B, K)
        return (-F.logsigmoid(s_pos) - F.logsigmoid(-s_neg).sum(-1)).mean()

cbow_ds = CBOWDataset(TOKENIZED, word2idx, CFG["window"])
cbow_loader = DataLoader(cbow_ds, batch_size=CFG["batch_size"], shuffle=True)
cbow_model = CBOW_NS(V, CFG["emb_dim"]).to(DEVICE)
opt_cbow = torch.optim.Adam(cbow_model.parameters(), lr=CFG["lr"])

cbow_losses = []
for epoch in range(CFG["cbow_epochs"]):
    total = 0.0
    for ctx, center in cbow_loader:
        ctx, center = ctx.to(DEVICE), center.to(DEVICE)
        negs = torch.multinomial(noise_probs_t, center.size(0) * K, replacement=True)
        negs = negs.view(center.size(0), K).to(DEVICE)
        loss = cbow_model(ctx, center, negs)
        opt_cbow.zero_grad(); loss.backward(); opt_cbow.step()
        total += loss.item()
    cbow_losses.append(total / len(cbow_loader))

W_cbow = cbow_model.in_emb.weight.detach().cpu().numpy()[:V]  # (V, d) — sin el slot de padding

plt.figure(figsize=(8, 4))
plt.plot(np.linspace(0, 1, len(sg_losses)), sg_losses, label="skip-gram ({} ej/epoch)".format(len(sg_dataset)), lw=2)
plt.plot(np.linspace(0, 1, len(cbow_losses)), cbow_losses, label="CBOW ({} ej/epoch)".format(len(cbow_ds)), lw=2)
plt.xlabel("progreso de entrenamiento (normalizado)"); plt.ylabel("pérdida NS media")
plt.title("Skip-gram vs CBOW: CBOW tiene ~{}x menos ejemplos por epoch".format(round(len(sg_dataset) / len(cbow_ds))))
plt.legend(); plt.tight_layout(); plt.show()
print("Nota: las pérdidas no son directamente comparables (tareas distintas),")
print("pero sí lo es el NÚMERO de ejemplos: skip-gram extrae más señal del mismo corpus.")


<a id="s4"></a>
## 4. GloVe: factorización ponderada de co-ocurrencias

Word2Vec recorre el corpus par por par (*método predictivo*). **GloVe** (Pennington et al., 2014) parte de la matriz de co-ocurrencia global $X$ (*método de conteo*) y la factoriza.

### Derivación de la función objetivo

**Punto de partida:** la semántica vive en las **razones** de probabilidades de co-ocurrencia. Ejemplo del paper: para $w \in \{ \text{hielo}, \text{vapor} \}$, la razón $P(k|\text{hielo}) / P(k|\text{vapor})$ es grande si $k = \text{sólido}$, pequeña si $k = \text{gas}$ y $\approx 1$ si $k$ es irrelevante (*agua*, *moda*). Buscamos vectores tales que una función de ellos capture esas razones; tras imponer linealidad y simetría palabra↔contexto, el paper llega a:

$$\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j = \log X_{ij}$$

Es decir: **el producto punto debe aproximar el log de la co-ocurrencia** (los sesgos $b_i, \tilde b_j$ absorben las frecuencias marginales de cada palabra). Resolverlo exactamente es imposible; se plantea como mínimos cuadrados **ponderados**:

$$J = \sum_{i,j:\, X_{ij} > 0} f(X_{ij}) \left( \mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

### La función de peso $f(x)$

$$f(x) = \begin{cases} (x / x_{\max})^{\alpha} & x < x_{\max} \\ 1 & x \ge x_{\max} \end{cases} \qquad \alpha = 0.75, \quad x_{\max} = 100$$

Tres propiedades deliberadas: (1) $f(0) = 0$ — los pares nunca vistos **no contribuyen** (la suma corre solo sobre entradas no nulas → eficiencia); (2) creciente — pares más frecuentes son evidencia más confiable; (3) **saturada** en $x_{\max}$ — "el la" no debe dominar el objetivo. ¿Te suena el exponente $0.75$? Es la misma compresión de frecuencias que el negative sampling.

### La conexión con PMI (Levy & Goldberg, 2014)

Resultado clave que unifica todo el notebook: **SGNS es implícitamente una factorización de matriz**. En el óptimo, los vectores de skip-gram con $K$ negativos satisfacen:

$$\mathbf{u}_c^\top \mathbf{v}_w = \text{PMI}(w, c) - \log K$$

Es decir, Word2Vec factoriza (aproximadamente) la matriz **PMI desplazada** — la misma matriz PPMI de la sección 1, menos una constante. GloVe factoriza $\log X_{ij}$ con sesgos libres — si los sesgos aprenden los marginales ($b_i \approx \log X_i$), su objetivo también se parece a PMI. **Conteo y predicción son dos caminos a la misma estructura.**


In [ ]:
# ============================================================
# GloVe simplificado sobre la matriz de co-ocurrencia de la sección 1
# ============================================================
class GloVe(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.w = nn.Embedding(vocab_size, emb_dim)        # w_i,  (V, d)
        self.w_tilde = nn.Embedding(vocab_size, emb_dim)  # w~_j, (V, d)
        self.b = nn.Embedding(vocab_size, 1)              # b_i,  (V, 1)
        self.b_tilde = nn.Embedding(vocab_size, 1)        # b~_j, (V, 1)
        for emb in [self.w, self.w_tilde]:
            nn.init.uniform_(emb.weight, -0.5 / emb_dim, 0.5 / emb_dim)
        for emb in [self.b, self.b_tilde]:
            nn.init.zeros_(emb.weight)

    def forward(self, i, j, log_x, weight):
        # i, j: (N,) índices | log_x, weight: (N,)
        pred = (self.w(i) * self.w_tilde(j)).sum(-1) + self.b(i).squeeze(-1) + self.b_tilde(j).squeeze(-1)  # (N,)
        return (weight * (pred - log_x) ** 2).mean()

# Entradas no nulas de la matriz de co-ocurrencia (sección 1)
ii, jj = np.nonzero(X_cooc)
x_vals = X_cooc[ii, jj]  # (N_nz,)

X_MAX, ALPHA = 30.0, 0.75  # x_max menor que 100: nuestro corpus es pequeño
f_weight = np.where(x_vals < X_MAX, (x_vals / X_MAX) ** ALPHA, 1.0)  # f(X_ij), (N_nz,)

t_i = torch.tensor(ii, dtype=torch.long, device=DEVICE)
t_j = torch.tensor(jj, dtype=torch.long, device=DEVICE)
t_logx = torch.tensor(np.log(x_vals), dtype=torch.float32, device=DEVICE)
t_w = torch.tensor(f_weight, dtype=torch.float32, device=DEVICE)

glove = GloVe(V, CFG["emb_dim"]).to(DEVICE)
opt_glove = torch.optim.Adam(glove.parameters(), lr=0.05)

glove_losses = []
for epoch in range(CFG["glove_epochs"]):  # full-batch: caben todas las entradas no nulas
    loss = glove(t_i, t_j, t_logx, t_w)
    opt_glove.zero_grad(); loss.backward(); opt_glove.step()
    glove_losses.append(loss.item())

# Embedding final: w + w~ (recomendación del paper: suma de ambas matrices)
W_glove = (glove.w.weight + glove.w_tilde.weight).detach().cpu().numpy()  # (V, d)
print("Entradas no nulas factorizadas: {:,}".format(len(x_vals)))
print("Pérdida GloVe: {:.4f} -> {:.4f}".format(glove_losses[0], glove_losses[-1]))


In [ ]:
# ============================================================
# Visualización: f(x) y curva de pérdida de GloVe
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

xs = np.linspace(0, 60, 300)
fx = np.where(xs < X_MAX, (xs / X_MAX) ** ALPHA, 1.0)
axes[0].plot(xs, fx, lw=2, color="darkorange")
axes[0].axvline(X_MAX, ls="--", color="gray", label="x_max = {}".format(int(X_MAX)))
axes[0].set_xlabel("co-ocurrencia X_ij"); axes[0].set_ylabel("f(X_ij)")
axes[0].set_title("Función de peso de GloVe: creciente y saturada")
axes[0].legend()

axes[1].plot(glove_losses, lw=2, color="seagreen")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("J (ponderada)")
axes[1].set_title("GloVe — pérdida de la factorización")
plt.tight_layout(); plt.show()

# Chequeo rápido: el producto punto aprendido aproxima log(X_ij)
with torch.no_grad():
    pred = (glove.w(t_i) * glove.w_tilde(t_j)).sum(-1) + glove.b(t_i).squeeze(-1) + glove.b_tilde(t_j).squeeze(-1)
corr = np.corrcoef(pred.cpu().numpy(), t_logx.cpu().numpy())[0, 1]
print("Correlación entre w_i·w~_j + b_i + b~_j y log(X_ij): {:.3f}".format(corr))
assert corr > 0.8, "La factorización debería ajustar bien las co-ocurrencias"
print("La factorización reconstruye las log co-ocurrencias ✓")


<a id="s5"></a>
## 5. FastText: n-gramas de subpalabras

Word2Vec y GloVe tratan cada palabra como un átomo: *transformador* y *transformadores* tienen vectores **independientes**, y una palabra no vista (OOV, *out-of-vocabulary*) no tiene vector en absoluto. Para idiomas morfológicamente ricos como el español (conjugaciones, plurales, diminutivos) esto desperdicia muchísima estructura.

**FastText** (Bojanowski et al., 2017) representa cada palabra como la **suma de los embeddings de sus n-gramas de caracteres**. La palabra se delimita con `<` y `>` y se extraen los n-gramas de longitud 3 a 6. Para *donde* con $n=3$:

$$G_{\text{<donde>}} = \{ \texttt{<do}, \texttt{don}, \texttt{ond}, \texttt{nde}, \texttt{de>} \} \cup \{ \texttt{<donde>} \}$$

### Derivación: el vector como suma de subpalabras

El vector de la palabra $w$ se define como:

$$\mathbf{v}_w = \sum_{g \in G_w} \mathbf{z}_g$$

donde $\mathbf{z}_g$ es el embedding del n-grama $g$ (almacenados en una tabla hasheada de tamaño fijo). La pérdida es **exactamente la SGNS de la sección 2**, solo que el score se reescribe:

$$s(w, c) = \mathbf{v}_w^\top \mathbf{u}_c = \sum_{g \in G_w} \mathbf{z}_g^\top \mathbf{u}_c$$

Por linealidad, el gradiente $\partial \mathcal{L} / \partial \mathbf{v}_w$ se **reparte idéntico a cada n-grama**: $\partial \mathcal{L} / \partial \mathbf{z}_g = \partial \mathcal{L} / \partial \mathbf{v}_w$ para todo $g \in G_w$. Cada ocurrencia de *transformador* también entrena los n-gramas `transf`, `forma`, `ador>`...

### Por qué resuelve OOV y morfología

- **Morfología:** *transformador* y *transformadores* comparten casi todos sus n-gramas → vectores casi idénticos **por construcción**, sin necesidad de ver ambas formas muchas veces.
- **OOV:** una palabra jamás vista (*subestaciones*, un typo como *voltage*) se descompone en n-gramas, muchos de los cuales sí fueron entrenados → se sintetiza un vector razonable donde Word2Vec simplemente falla con `KeyError`.


In [ ]:
# ============================================================
# N-gramas de subpalabras desde cero
# ============================================================
def char_ngrams(word, n_min=3, n_max=6):
    """N-gramas de caracteres con delimitadores < > (como FastText)."""
    w = "<" + word + ">"
    grams = set()
    for n in range(n_min, n_max + 1):
        for i in range(len(w) - n + 1):
            grams.add(w[i:i + n])
    grams.add(w)  # la palabra completa también es un 'n-grama'
    return grams

g_sing = char_ngrams("transformador")
g_plur = char_ngrams("transformadores")
shared = g_sing & g_plur

print("N-gramas de 'transformador':   {}".format(len(g_sing)))
print("N-gramas de 'transformadores': {}".format(len(g_plur)))
print("Compartidos: {} ({:.0%} del singular)".format(len(shared), len(shared) / len(g_sing)))
print()
print("Algunos n-gramas compartidos:", sorted(shared)[:12])
print()

# Similitud de Jaccard entre conjuntos de n-gramas: proxy de cercanía morfológica
def jaccard(a, b):
    return len(a & b) / len(a | b)

pares = [("transformador", "transformadores"), ("transformador", "transformacion"),
         ("voltaje", "voltajes"), ("transformador", "mongodb")]
print("Solapamiento de n-gramas (Jaccard):")
for a, b in pares:
    print("  {:<16} vs {:<17}: {:.2f}".format(a, b, jaccard(char_ngrams(a), char_ngrams(b))))
print()
print("Palabras morfológicamente emparentadas comparten n-gramas -> comparten vector.")


In [ ]:
# ============================================================
# FastText con gensim sobre nuestro corpus (try/except)
# ============================================================
FT_AVAILABLE = False
try:
    from gensim.models import FastText

    ft = FastText(
        sentences=TOKENIZED, vector_size=CFG["emb_dim"], window=CFG["window"],
        min_count=1, sg=1, negative=CFG["neg_k"], epochs=60,
        min_n=3, max_n=6, seed=SEED, workers=1,  # workers=1: reproducible
    )
    FT_AVAILABLE = True

    print("FastText entrenado: {} palabras, dim {}".format(len(ft.wv), ft.wv.vector_size))
    print()

    # 1) Morfología: singular vs plural ('transformadores' NO está en el corpus)
    sim = ft.wv.similarity("transformador", "transformadores")
    print("similitud('transformador', 'transformadores') = {:.3f}".format(sim))
    print("   ('transformadores' es OOV: su vector se sintetiza desde n-gramas)")
    print()

    # 2) OOV total: typo y variante
    for oov in ["subestaciones", "voltage", "gradientes"]:
        vecinos = ft.wv.most_similar(oov, topn=3)
        print("OOV '{}' -> vecinos: {}".format(oov, [(w, round(s, 2)) for w, s in vecinos]))
    print()
    print("Word2Vec habría lanzado KeyError con estas palabras; FastText les da un vector útil.")
except Exception as e:
    print("gensim/FastText no disponible o falló: {}".format(e))
    print("Continuamos con los modelos propios; la sección es ilustrativa.")


<a id="s6"></a>
## 6. Similitud semántica: coseno, euclidiana y producto punto

Tres formas de comparar vectores $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$, relacionadas por la identidad $\|\mathbf{a} - \mathbf{b}\|^2 = \|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - 2\,\mathbf{a}^\top\mathbf{b}$:

| Medida | Fórmula | Sensible a la norma | Rango |
|---|---|---|---|
| Producto punto | $\mathbf{a}^\top \mathbf{b}$ | **Sí** (a ambas) | $(-\infty, \infty)$ |
| Distancia euclidiana | $\|\mathbf{a} - \mathbf{b}\|_2$ | **Sí** | $[0, \infty)$ |
| Similitud coseno | $\dfrac{\mathbf{a}^\top \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$ | **No** (solo ángulo) | $[-1, 1]$ |

### El efecto de la norma — por qué casi siempre se usa coseno

En Word2Vec/GloVe, la **norma del vector crece con la frecuencia** de la palabra (más updates → vectores más largos). Entonces:

- **Producto punto** favorece sistemáticamente a palabras frecuentes: *de* puede tener mayor $\mathbf{a}^\top\mathbf{b}$ con todo, solo por su norma.
- **Euclidiana** penaliza pares donde una palabra es frecuente y la otra rara, aunque apunten en la misma dirección.
- **Coseno** compara solo **direcciones** → mide afinidad semántica independiente de la frecuencia. Es el estándar para embeddings de palabras.

¿Cuándo usar cada una? **Coseno**: similitud semántica entre palabras/documentos (default). **Producto punto**: cuando la magnitud codifica algo útil — p. ej. en retrieval con embeddings entrenados para ello, o dentro de la atención del transformer ($QK^\top$). **Euclidiana**: clustering (KMeans la asume) y cuando los vectores están normalizados — en ese caso es equivalente al coseno: $\|\mathbf{a}-\mathbf{b}\|^2 = 2(1 - \cos\theta)$.


In [ ]:
# ============================================================
# Las tres medidas, con verificación de la identidad que las une
# ============================================================
def dot_sim(a, b):
    return float(np.dot(a, b))

def euclidean_dist(a, b):
    return float(np.linalg.norm(a - b))

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

a = W_sg[word2idx["voltaje"]]      # (d,)
b = W_sg[word2idx["corriente"]]    # (d,)

# Verificación: ||a-b||² = ||a||² + ||b||² − 2 a·b
lhs = euclidean_dist(a, b) ** 2
rhs = np.dot(a, a) + np.dot(b, b) - 2 * np.dot(a, b)
assert abs(lhs - rhs) < 1e-4, "La identidad euclidiana-producto punto no cuadra"
print("Identidad ||a-b||² = ||a||² + ||b||² − 2a·b verificada ✓")

# Verificación: con vectores normalizados, euclidiana² = 2(1 − cos)
an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b)
assert abs(euclidean_dist(an, bn) ** 2 - 2 * (1 - cosine_sim(a, b))) < 1e-4
print("Con vectores normalizados: ||a−b||² = 2(1 − cosθ) ✓")
print()

# --- Efecto de la norma: frecuencia infla el producto punto ---
norms = np.linalg.norm(W_sg, axis=1)  # (V,)
plt.figure(figsize=(7.5, 4))
plt.scatter(freqs, norms, s=14, alpha=0.5)
for w in ["de", "la", "transformador", "subestacion", "voltaje"]:
    i = word2idx[w]
    plt.annotate(w, (freqs[i], norms[i]), fontsize=9, fontweight="bold")
plt.xscale("log")
plt.xlabel("frecuencia de la palabra (log)"); plt.ylabel("||v_w||")
plt.title("La norma del embedding correlaciona con la frecuencia\n(por eso el producto punto favorece palabras frecuentes)")
plt.tight_layout(); plt.show()

corr_fn = np.corrcoef(np.log(freqs), norms)[0, 1]
print("Correlación log(frecuencia) vs norma: {:.2f}".format(corr_fn))


In [ ]:
# ============================================================
# nearest_neighbors() y tablas de vecinos para términos técnicos
# ============================================================
import pandas as pd

def nearest_neighbors(word, W, word2idx, idx2word, k=5, metric="cosine"):
    """Devuelve los k vecinos más cercanos de `word` en la matriz W (V, d)."""
    if word not in word2idx:
        return []
    q = W[word2idx[word]]  # (d,)
    if metric == "cosine":
        Wn = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-12)
        scores = Wn @ (q / (np.linalg.norm(q) + 1e-12))  # (V,)
        order = np.argsort(-scores)
    elif metric == "dot":
        scores = W @ q
        order = np.argsort(-scores)
    else:  # euclidean
        scores = -np.linalg.norm(W - q, axis=1)
        order = np.argsort(-scores)
    out = []
    for i in order:
        if idx2word[i] != word:
            out.append((idx2word[i], float(scores[i])))
        if len(out) == k:
            break
    return out

# Tabla comparativa: vecinos por coseno en skip-gram, GloVe y PPMI
query_terms = ["transformador", "voltaje", "subestacion", "gradiente", "neuronal", "pipeline"]
rows = {}
for term in query_terms:
    rows[term] = {
        "skip-gram": ", ".join(w for w, _ in nearest_neighbors(term, W_sg, word2idx, idx2word, k=4)),
        "GloVe": ", ".join(w for w, _ in nearest_neighbors(term, W_glove, word2idx, idx2word, k=4)),
        "PPMI (disperso)": ", ".join(w for w, _ in nearest_neighbors(term, X_ppmi, word2idx, idx2word, k=4)),
    }
df_nn = pd.DataFrame(rows).T
df_nn.index.name = "consulta"
print("Vecinos más cercanos (coseno) según cada modelo:")
display(df_nn)

# Coseno vs producto punto: el sesgo de frecuencia en acción
print()
print("Vecinos de 'transformador' por COSENO vs PRODUCTO PUNTO (skip-gram):")
df_metric = pd.DataFrame({
    "coseno": [w for w, _ in nearest_neighbors("transformador", W_sg, word2idx, idx2word, k=6)],
    "producto punto": [w for w, _ in nearest_neighbors("transformador", W_sg, word2idx, idx2word, k=6, metric="dot")],
})
display(df_metric)
print("El producto punto tiende a colar palabras frecuentes; el coseno respeta la dirección semántica.")


<a id="s7"></a>
## 7. Espacios vectoriales y aritmética de embeddings

El resultado más famoso de Word2Vec: las **relaciones semánticas son (aproximadamente) traslaciones** en el espacio de embeddings.

$$\mathbf{v}_{\text{rey}} - \mathbf{v}_{\text{hombre}} + \mathbf{v}_{\text{mujer}} \approx \mathbf{v}_{\text{reina}}$$

La dirección $\mathbf{v}_{\text{rey}} - \mathbf{v}_{\text{hombre}}$ captura "realeza sin género"; al sumarle *mujer* aterrizamos cerca de *reina*. En nuestro dominio: $\text{voltaje} : \text{corriente} :: \text{presion} : \text{caudal}$ (la analogía hidráulica clásica de la ingeniería eléctrica — por eso sembramos en el corpus oraciones paralelas sobre ambas).

### 3CosAdd

El método estándar para resolver "$a$ es a $b$ como $a^*$ es a ¿?" (Mikolov 2013; formalizado por Levy & Goldberg):

$$b^* = \arg\max_{x \in V \setminus \{a, b, a^*\}} \cos\big(\mathbf{v}_x,\; \mathbf{v}_b - \mathbf{v}_a + \mathbf{v}_{a^*}\big)$$

Detalle crucial y a menudo omitido: se **excluyen las tres palabras de la consulta** del argmax — sin esa exclusión, la respuesta casi siempre sería $b$ o $a^*$ (el vector consulta queda cerca de sus sumandos).

### Cuándo fallan las analogías

1. **Corpus insuficiente:** la regularidad lineal emerge de *millones* de contextos; con nuestro corpus diminuto la mayoría de analogías fallará (lo veremos — y por eso probamos también con un modelo preentrenado).
2. **Relaciones no lineales:** las analogías 1-a-1 funcionan para relaciones "paralelas" (género, capital-país, morfología); fallan con polisemia, hiperonimia, o relaciones muchos-a-muchos.
3. **El truco de la exclusión:** parte del éxito reportado depende de excluir las palabras de la consulta; sin exclusión la precisión cae drásticamente (Linzen, 2016).
4. **Sesgo del corpus:** las direcciones también codifican estereotipos presentes en los datos (Bolukbasi et al., 2016: *computer programmer − man + woman ≈ homemaker*). La aritmética hereda lo bueno y lo malo del corpus.


In [ ]:
# ============================================================
# 3CosAdd desde cero, probado en el modelo propio
# ============================================================
def cos_add_3(a, b, a_star, W, word2idx, idx2word, topn=5):
    """Resuelve a:b :: a_star:? con 3CosAdd, excluyendo las palabras de la consulta."""
    for w in (a, b, a_star):
        if w not in word2idx:
            return None
    Wn = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-12)  # (V, d) normalizada
    q = Wn[word2idx[b]] - Wn[word2idx[a]] + Wn[word2idx[a_star]]  # (d,)
    q = q / (np.linalg.norm(q) + 1e-12)
    scores = Wn @ q  # (V,)
    excluded = {word2idx[a], word2idx[b], word2idx[a_star]}
    order = [i for i in np.argsort(-scores) if i not in excluded]
    return [(idx2word[i], float(scores[i])) for i in order[:topn]]

print("Analogia tecnica: voltaje : corriente :: presion : ?")
print("(esperado: caudal — la analogia hidraulica sembrada en el corpus)")
print()
for nombre, W_m in [("skip-gram", W_sg), ("GloVe", W_glove)]:
    res = cos_add_3("voltaje", "corriente", "presion", W_m, word2idx, idx2word)
    print("{:<10}: {}".format(nombre, [(w, round(s, 3)) for w, s in res[:3]]))

rank = None
res_sg = cos_add_3("voltaje", "corriente", "presion", W_sg, word2idx, idx2word, topn=V)
for pos, (w, _) in enumerate(res_sg, start=1):
    if w == "caudal":
        rank = pos
        break
print()
if rank is not None and rank <= 5:
    print("'caudal' aparece en el puesto {} — ¡el corpus paralelo funcionó!".format(rank))
elif rank is not None:
    print("'caudal' quedó en el puesto {} de {}: con ~130 oraciones la regularidad".format(rank, V))
    print("lineal apenas emerge. Las analogías necesitan corpus masivos (ver celda siguiente).")
else:
    print("'caudal' no está en el ranking (revisa el vocabulario).")


In [ ]:
# ============================================================
# Analogías con un modelo preentrenado (gensim downloader, try/except)
# ============================================================
# Descarga ~66 MB la primera vez. Si no hay red, hacemos fallback al modelo propio.
PRETRAINED = None
try:
    import gensim.downloader as gd
    PRETRAINED = gd.load("glove-wiki-gigaword-50")  # inglés, 50d, entrenado con 6B tokens
    print("Modelo preentrenado cargado: glove-wiki-gigaword-50 ({} palabras)".format(len(PRETRAINED)))
    print()

    print("king - man + woman = ?")
    for w, s in PRETRAINED.most_similar(positive=["king", "woman"], negative=["man"], topn=3):
        print("   {:<12} {:.3f}".format(w, s))
    print()
    print("voltage : current :: pressure : ?")
    for w, s in PRETRAINED.most_similar(positive=["current", "pressure"], negative=["voltage"], topn=3):
        print("   {:<12} {:.3f}".format(w, s))
    print()
    print("paris - france + germany = ?")
    for w, s in PRETRAINED.most_similar(positive=["paris", "germany"], negative=["france"], topn=3):
        print("   {:<12} {:.3f}".format(w, s))
    print()
    print("Con 6B de tokens, la estructura lineal SÍ emerge: 'queen', 'flow', 'berlin'.")
except Exception as e:
    print("No se pudo descargar el modelo preentrenado: {}".format(type(e).__name__))
    print("Fallback: seguimos con el modelo propio (las analogías serán ruidosas).")
    print("Para reproducir esta celda se necesita conexión a internet (~66 MB).")


<a id="s8"></a>
## 8. Embeddings contextuales vs estáticos: el problema de la polisemia

Todo lo anterior asigna **un único vector por palabra**, sin importar el contexto. Eso colapsa los sentidos de las palabras polisémicas:

- **transformador**: ¿la máquina eléctrica de la subestación... o la arquitectura *transformer* de redes neuronales (como se usa a veces en textos en español)?
- **banco**: ¿el asiento del parque, la institución financiera o el *banco de datos*?
- **corriente**: ¿eléctrica, de agua, o "al corriente"?

Un embedding estático de *banco* es un **promedio ponderado de todos sus sentidos** en el corpus — no representa bien ninguno.

### La solución contextual

Los modelos tipo BERT/GPT calculan el embedding de cada token **en función de toda la oración**: el vector de *banco* en "me senté en el banco" y en "el banco aprobó el crédito" son **distintos**, porque la atención mezcla información de los vecinos en cada capa. Formalmente, en la capa $\ell$:

$$\mathbf{h}^{(\ell)}_t = f_\ell\big(\mathbf{h}^{(\ell-1)}_1, \dots, \mathbf{h}^{(\ell-1)}_T\big)$$

— el vector del token $t$ depende de **todos** los tokens (en BERT, bidireccional). La capa 0 es (casi) estática: embedding de token + posición. A medida que subimos de capa, el vector incorpora más contexto.

**Experimento:** extraeremos el vector de *transformador*/*transformer* y *banco* en varias oraciones con un modelo multilingüe pequeño, capa por capa, y mediremos si las ocurrencias **del mismo sentido** se parecen más entre sí que las de **sentidos distintos**.


In [ ]:
# ============================================================
# Cargar modelo HF multilingüe pequeño (try/except con fallback)
# ============================================================
HF_AVAILABLE = False
hf_model, hf_tokenizer = None, None
try:
    from transformers import AutoModel, AutoTokenizer

    HF_NAME = "distilbert-base-multilingual-cased"  # 6 capas, ~135M params, es+en
    hf_tokenizer = AutoTokenizer.from_pretrained(HF_NAME)
    hf_model = AutoModel.from_pretrained(HF_NAME, output_hidden_states=True)
    hf_model = hf_model.to(DEVICE).eval()
    HF_AVAILABLE = True
    n_layers = hf_model.config.num_hidden_layers
    print("Modelo cargado: {} ({} capas, hidden={})".format(HF_NAME, n_layers, hf_model.config.dim))
except Exception as e:
    print("No se pudo cargar el modelo HF: {}".format(type(e).__name__))
    print("Las celdas contextuales se saltarán con un mensaje; el resto del notebook funciona.")

def get_word_vectors(sentence, target, layer=-1):
    """Vector contextual de `target` en `sentence` para cada capa pedida.
    Promedia los subtokens que cubren la palabra. Devuelve (n_capas+1, hidden)."""
    enc = hf_tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]  # (T, 2) spans de caracteres
    start = sentence.lower().find(target.lower())
    assert start >= 0, "'{}' no está en la oración".format(target)
    end = start + len(target)
    token_ids = [i for i, (s, e) in enumerate(offsets.tolist())
                 if not (s == e) and s < end and e > start]  # subtokens que solapan la palabra
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = hf_model(**enc)
    # hidden_states: tupla de (n_capas+1) tensores (1, T, hidden); índice 0 = embeddings
    layers = torch.stack(out.hidden_states, dim=0)[:, 0]  # (L+1, T, hidden)
    return layers[:, token_ids, :].mean(dim=1).cpu().numpy()  # (L+1, hidden)


In [ ]:
# ============================================================
# El mismo token, vectores distintos según el contexto
# ============================================================
# Ocurrencias de palabras polisémicas en contextos que fuerzan un sentido:
POLY_CASES = [
    # (etiqueta, palabra objetivo, oración, sentido)
    ("transformador/eléctrico-1", "transformador", "el transformador de potencia de la subestacion eleva el voltaje de la red electrica", "electrico"),
    ("transformador/eléctrico-2", "transformador", "el transformador se refrigera con aceite para soportar la corriente de la subestacion", "electrico"),
    ("transformador/NN-1", "transformador", "el transformador es la arquitectura de red neuronal basada en atencion que usan los modelos de lenguaje", "red neuronal"),
    ("transformador/NN-2", "transformador", "entrenamos un transformador con descenso de gradiente para predecir el siguiente token del texto", "red neuronal"),
    ("banco/asiento", "banco", "me sente en el banco de madera del parque a descansar", "asiento"),
    ("banco/finanzas-1", "banco", "el banco aprobo el credito hipotecario con una tasa de interes baja", "finanzas"),
    ("banco/finanzas-2", "banco", "el banco central subio la tasa de interes para controlar la inflacion", "finanzas"),
    ("banco/datos", "banco", "el banco de datos almacena las mediciones de los sensores del sistema", "datos"),
]

if HF_AVAILABLE:
    poly_vecs = []   # lista de (L+1, hidden)
    poly_labels = []
    poly_senses = []
    for label, target, sent, sense in POLY_CASES:
        poly_vecs.append(get_word_vectors(sent, target))
        poly_labels.append(label)
        poly_senses.append(sense)
    poly_vecs = np.stack(poly_vecs)  # (n_casos, L+1, hidden)
    print("Vectores extraídos: {} ocurrencias x {} capas x {} dims".format(*poly_vecs.shape))
    print()
    # Demostración puntual: mismo token, vectores distintos
    v_elec = poly_vecs[0, -1]  # transformador eléctrico, última capa
    v_nn = poly_vecs[2, -1]    # transformador NN, última capa
    v_elec2 = poly_vecs[1, -1]
    print("Última capa — cos(transformador_eléctrico_1, transformador_eléctrico_2) = {:.3f}".format(
        cosine_sim(v_elec, v_elec2)))
    print("Última capa — cos(transformador_eléctrico_1, transformador_NN_1)        = {:.3f}".format(
        cosine_sim(v_elec, v_nn)))
    print()
    print("¡El MISMO token tiene vectores distintos según el contexto!")
    print("Un embedding estático habría dado similitud = 1.000 en ambos casos.")
else:
    print("Modelo HF no disponible: se omite el experimento contextual.")


In [ ]:
# ============================================================
# Heatmap: similitud coseno entre todas las ocurrencias (última capa)
# ============================================================
if HF_AVAILABLE:
    last = poly_vecs[:, -1, :]  # (n_casos, hidden)
    last_n = last / np.linalg.norm(last, axis=1, keepdims=True)
    sim_matrix = last_n @ last_n.T  # (n, n)

    plt.figure(figsize=(8.5, 7))
    sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap="RdYlGn", center=0.6,
                xticklabels=poly_labels, yticklabels=poly_labels,
                annot_kws={"size": 8})
    plt.title("Similitud coseno entre ocurrencias (última capa)\nBloques = sentidos: el contexto separa la polisemia")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

    print("Lectura: las ocurrencias del MISMO sentido (bloques diagonales) se parecen más")
    print("entre sí que las de sentidos distintos — exactamente lo que un estático no puede hacer.")
else:
    print("Modelo HF no disponible: se omite el heatmap contextual.")


In [ ]:
# ============================================================
# ¿Qué capa es más 'semántica'? Separación intra/inter-sentido por capa
# ============================================================
if HF_AVAILABLE:
    n_layers_total = poly_vecs.shape[1]  # L+1 (capa 0 = embeddings estáticos + posición)
    intra_means, inter_means = [], []
    for layer in range(n_layers_total):
        vecs = poly_vecs[:, layer, :]
        vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
        sims = vecs @ vecs.T
        intra, inter = [], []
        for i in range(len(poly_senses)):
            for j in range(i + 1, len(poly_senses)):
                # solo comparamos ocurrencias de la MISMA palabra superficial
                same_word = poly_labels[i].split("/")[0] == poly_labels[j].split("/")[0]
                if not same_word:
                    continue
                (intra if poly_senses[i] == poly_senses[j] else inter).append(sims[i, j])
        intra_means.append(np.mean(intra))
        inter_means.append(np.mean(inter))

    intra_means, inter_means = np.array(intra_means), np.array(inter_means)
    gap = intra_means - inter_means  # separación de sentidos por capa

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    layers_x = np.arange(n_layers_total)
    axes[0].plot(layers_x, intra_means, "o-", label="mismo sentido (intra)", lw=2)
    axes[0].plot(layers_x, inter_means, "s-", label="sentidos distintos (inter)", lw=2)
    axes[0].set_xlabel("capa (0 = embeddings de entrada)"); axes[0].set_ylabel("coseno medio")
    axes[0].set_title("Similitud entre ocurrencias del mismo token"); axes[0].legend()

    axes[1].bar(layers_x, gap, color="teal")
    axes[1].set_xlabel("capa"); axes[1].set_ylabel("intra − inter")
    axes[1].set_title("Separación de sentidos por capa\n(mayor = la capa distingue mejor la polisemia)")
    plt.tight_layout(); plt.show()

    best = int(np.argmax(gap))
    print("En la capa 0 (estática) la separación es ~{:.3f}: el token es el token.".format(gap[0]))
    print("La separación máxima ocurre en la capa {} ({:.3f}).".format(best, gap[best]))
else:
    print("Modelo HF no disponible: se omite el análisis por capas.")


### Lectura del experimento

Hallazgos típicos (consistentes con la literatura de *BERTology*, p. ej. Tenney et al. 2019; Ethayarajh 2019):

- **Capa 0** es casi estática: el mismo token tiene casi el mismo vector en cualquier contexto (la única diferencia es la codificación posicional). La separación intra/inter ≈ 0.
- **Capas intermedias-altas** son las más "semánticas": maximizan la separación entre sentidos. En BERT-base suelen ser las capas medias-altas las que mejor capturan sentido léxico; las últimas se especializan en el objetivo de preentrenamiento (masked LM).
- **Anisotropía:** en capas altas, *todos* los cosenos tienden a ser altos (los vectores ocupan un cono estrecho del espacio); por eso lo informativo es la **diferencia** intra−inter, no los valores absolutos.

**Conclusión central del notebook:** los embeddings estáticos asignan un punto por palabra; los contextuales asignan **una nube por palabra**, con un punto por ocurrencia, organizada por sentido. Por eso BERT/GPT desplazaron a Word2Vec en casi toda tarea — y aún así, los estáticos siguen siendo útiles: baratos, interpretables y suficientes cuando no hay ambigüedad (búsqueda de vocabulario técnico, features tabulares, sistemas con restricciones de cómputo).


<a id="s9"></a>
## 9. Embeddings dentro del transformer

En nuestro GPT (notebooks 05-06) los embeddings no son un producto final sino la **primera capa** del modelo. Tres piezas:

### 9.1 La matriz de embedding como lookup

$W_{emb} \in \mathbb{R}^{|V| \times d_{model}}$. Para un token con índice $i$, su embedding es la fila $i$. Matemáticamente equivale a multiplicar por el one-hot — $\mathbf{e}_i^\top W_{emb} = W_{emb}[i]$ — pero se implementa como **indexación directa** ($O(d)$ en lugar de $O(|V| \cdot d)$): `nn.Embedding` es exactamente eso. El círculo se cierra: el one-hot "inútil" de la sección 1 vive implícitamente dentro de cada LLM, solo que nunca se materializa.

### 9.2 Tied embeddings (weight tying)

La capa de salida del LM proyecta $\mathbf{h}_t \in \mathbb{R}^{d}$ a logits sobre el vocabulario: $\text{logits} = \mathbf{h}_t W_{out}^\top$ con $W_{out} \in \mathbb{R}^{|V| \times d}$ — la **misma forma** que $W_{emb}$. *Weight tying* (Press & Wolf, 2017) impone $W_{out} = W_{emb}$:

- **Ahorro:** $|V| \times d$ parámetros (en GPT-2 small: $50257 \times 768 \approx 38.6$M, ¡un 31% del modelo!).
- **Justificación:** ambas matrices aprenden representaciones de las *mismas* palabras (¿qué token es? / ¿qué token sigue?); compartirlas actúa como regularización y suele **mejorar** la perplejidad.
- El logit del token $j$ es $\mathbf{h}_t^\top \mathbf{w}_j$: predecir el siguiente token = **buscar el embedding más alineado** con el estado oculto (¡un producto punto, sección 6!).

### 9.3 ¿Qué aprende la matriz posicional?

La atención es invariante a permutaciones: sin información de posición, "perro muerde hombre" = "hombre muerde perro". Se suma una matriz $W_{pos} \in \mathbb{R}^{T_{max} \times d}$: $\mathbf{x}_t = W_{emb}[i_t] + W_{pos}[t]$. Cuando $W_{pos}$ es **aprendida** (BERT, GPT-2), no se le impone estructura... y aun así aprende una: posiciones cercanas terminan con vectores similares (una estructura suave, casi sinusoidal), porque la tarea lo exige. Lo verificaremos visualmente.


In [ ]:
# ============================================================
# Lookup = one-hot @ W_emb (verificado) + tied embeddings
# ============================================================
emb_layer = nn.Embedding(V, CFG["emb_dim"])  # W_emb: (V, d)
idx = torch.tensor([word2idx["transformador"]])

# Camino 1: lookup directo (lo que hace nn.Embedding)
v_lookup = emb_layer(idx)  # (1, d)

# Camino 2: one-hot @ W_emb (la formulación matemática)
one_hot_t = F.one_hot(idx, num_classes=V).float()  # (1, V)
v_matmul = one_hot_t @ emb_layer.weight            # (1, V) @ (V, d) -> (1, d)

assert torch.allclose(v_lookup, v_matmul, atol=1e-6)
print("nn.Embedding(idx) == one_hot @ W_emb ✓  (mismo resultado, costo O(d) vs O(V·d))")
print()

# --- Tied embeddings: compartir W_emb con la capa de salida ---
class TinyLMHead(nn.Module):
    """Cabeza de LM minimalista con weight tying."""
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)       # W_emb (V, d)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # W_out (V, d)
        self.lm_head.weight = self.emb.weight  # ¡tying! mismo tensor, no copia

    def forward(self, h):
        return self.lm_head(h)  # logits: h @ W_emb^T, (B, V)

tiny = TinyLMHead(V, CFG["emb_dim"])
untied_params = V * CFG["emb_dim"] * 2
tied_params = sum(p.numel() for p in tiny.parameters())
assert tiny.lm_head.weight.data_ptr() == tiny.emb.weight.data_ptr(), "deben compartir memoria"
print("Sin tying: {:>10,} parámetros (W_emb + W_out)".format(untied_params))
print("Con tying: {:>10,} parámetros (un solo tensor compartido) — ahorro del 50% en estas matrices".format(tied_params))
print()
print("El logit del token j es h·w_j: predecir el siguiente token es literalmente")
print("buscar el embedding más alineado con el estado oculto (producto punto, sección 6).")


In [ ]:
# ============================================================
# ¿Qué estructura aprende la matriz posicional?
# ============================================================
if HF_AVAILABLE:
    W_pos = hf_model.embeddings.position_embeddings.weight.detach().cpu().numpy()  # (T_max, d)
    fuente = "aprendida por " + HF_NAME
else:
    # Fallback: posicional sinusoidal (Vaswani et al. 2017) para ilustrar la estructura
    T_max, d = 512, 768
    pos = np.arange(T_max)[:, None]                       # (T_max, 1)
    div = np.exp(np.arange(0, d, 2) * (-math.log(10000.0) / d))  # (d/2,)
    W_pos = np.zeros((T_max, d))
    W_pos[:, 0::2] = np.sin(pos * div)
    W_pos[:, 1::2] = np.cos(pos * div)
    fuente = "sinusoidal (fallback sin modelo HF)"

n_show = 128
Wp = W_pos[:n_show]
Wp_n = Wp / (np.linalg.norm(Wp, axis=1, keepdims=True) + 1e-12)
pos_sim = Wp_n @ Wp_n.T  # (n_show, n_show)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
sns.heatmap(Wp[:, :64], ax=axes[0], cmap="RdBu_r", center=0, cbar_kws={"label": "valor"})
axes[0].set_xlabel("dimensión (primeras 64)"); axes[0].set_ylabel("posición")
axes[0].set_title("Matriz posicional ({})".format(fuente))

sns.heatmap(pos_sim, ax=axes[1], cmap="viridis", cbar_kws={"label": "coseno"})
axes[1].set_xlabel("posición"); axes[1].set_ylabel("posición")
axes[1].set_title("Similitud coseno entre posiciones:\nbanda diagonal = posiciones cercanas tienen vectores parecidos")
plt.tight_layout(); plt.show()

print("La banda brillante alrededor de la diagonal muestra que la matriz posicional")
print("codifica DISTANCIA: posiciones vecinas son similares y la similitud decae suavemente.")
print("Nadie le impuso esa estructura a la matriz aprendida: emerge del entrenamiento.")


<a id="s10"></a>
## 10. Sentence embeddings y buscador semántico

Para comparar **oraciones** (búsqueda, clustering, deduplicación) necesitamos un vector por oración. Dada la salida del encoder $H \in \mathbb{R}^{T \times d}$ (un vector por token, con máscara de atención $m \in \{0,1\}^T$), las tres estrategias clásicas de *pooling*:

| Estrategia | Fórmula | Notas |
|---|---|---|
| **Mean pooling** | $\mathbf{s} = \dfrac{\sum_t m_t \mathbf{h}_t}{\sum_t m_t}$ | El estándar robusto. ¡Cuidado: hay que **excluir el padding** con la máscara! |
| **CLS** | $\mathbf{s} = \mathbf{h}_{\text{[CLS]}}$ | El token especial de BERT. Sin fine-tuning específico suele ser **peor** que mean pooling (el preentrenamiento MLM no lo optimiza para representar la oración). |
| **Max pooling** | $s_j = \max_t \, (m_t \cdot h_{tj})$ | Por dimensión. Captura la activación más fuerte; útil cuando una palabra clave domina el significado. |

Con ellas montaremos un **mini buscador semántico**: codificar los ~130 documentos del corpus, codificar la *query*, y rankear por coseno. Es el corazón de cualquier sistema de búsqueda vectorial (y de RAG, que veremos en la Etapa 2).

Si el modelo HF no está disponible, el *fallback* es el promedio de nuestros embeddings skip-gram — más ruidoso, pero la mecánica es idéntica.


In [ ]:
# ============================================================
# Mean / CLS / Max pooling implementados desde cero
# ============================================================
def encode_sentences_hf(sentences, batch_size=16):
    """Codifica oraciones con el modelo HF. Devuelve dict con los 3 poolings."""
    means, clss, maxs = [], [], []
    for k in range(0, len(sentences), batch_size):
        batch = sentences[k:k + batch_size]
        enc = hf_tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        enc = {key: v.to(DEVICE) for key, v in enc.items()}
        with torch.no_grad():
            H = hf_model(**enc).last_hidden_state        # (B, T, d)
        mask = enc["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)

        # Mean pooling: suma enmascarada / número de tokens reales
        mean_pool = (H * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # (B, d)
        # CLS: primer token
        cls_pool = H[:, 0]                                            # (B, d)
        # Max pooling: -inf en el padding para que nunca gane
        H_masked = H.masked_fill(mask == 0, float("-inf"))
        max_pool = H_masked.max(dim=1).values                         # (B, d)

        means.append(mean_pool.cpu()); clss.append(cls_pool.cpu()); maxs.append(max_pool.cpu())
    return {
        "mean": torch.cat(means).numpy(),
        "cls": torch.cat(clss).numpy(),
        "max": torch.cat(maxs).numpy(),
    }

def encode_sentences_static(sentences):
    """Fallback: promedio de embeddings skip-gram propios (ignora OOV)."""
    out = np.zeros((len(sentences), W_sg.shape[1]), dtype=np.float32)
    for i, s in enumerate(sentences):
        ids = [word2idx[t] for t in tokenize(s) if t in word2idx]
        if ids:
            out[i] = W_sg[ids].mean(axis=0)
    return out

if HF_AVAILABLE:
    SENT_EMB = encode_sentences_hf(SENTENCES)        # dict con (N, 768) por estrategia
    SENT_VECS = SENT_EMB["mean"]                     # mean pooling como representación principal
    print("Codificadas {} oraciones con {} (mean/cls/max)".format(len(SENTENCES), HF_NAME))
else:
    SENT_VECS = encode_sentences_static(SENTENCES)   # (N, d)
    SENT_EMB = {"mean": SENT_VECS}
    print("Fallback estático: promedio de skip-gram propio. shape {}".format(SENT_VECS.shape))
print("Shape de la matriz de oraciones: {}".format(SENT_VECS.shape))


In [ ]:
# ============================================================
# Comparación de estrategias de pooling con pares de prueba
# ============================================================
test_pairs = [
    # (oración A, oración B, "esperamos similitud ALTA o BAJA")
    ("el transformador de la subestacion eleva el voltaje",
     "el transformador de potencia ajusta la tension en la subestacion", "ALTA"),
    ("la red neuronal aprende con descenso de gradiente",
     "el modelo profundo se entrena minimizando la perdida", "ALTA"),
    ("el transformador de la subestacion eleva el voltaje",
     "la lista por comprension construye la coleccion en python", "BAJA"),
    ("the transformer model uses attention layers",
     "la arquitectura transformer usa capas de atencion", "ALTA (cruce de idioma)"),
]

def encode_pair(a, b, strategy):
    if HF_AVAILABLE:
        embs = encode_sentences_hf([a, b])[strategy]
    else:
        embs = encode_sentences_static([a, b])
    return cosine_sim(embs[0], embs[1])

strategies = list(SENT_EMB.keys())
rows = []
for a, b, expect in test_pairs:
    row = {"esperado": expect}
    for st in strategies:
        row[st] = round(encode_pair(a, b, st), 3)
    row["par"] = a[:38] + "... / " + b[:38] + "..."
    rows.append(row)

df_pool = pd.DataFrame(rows).set_index("par")
print("Similitud coseno por estrategia de pooling:")
display(df_pool)
if HF_AVAILABLE:
    print()
    print("Típico: mean pooling separa mejor ALTA de BAJA; CLS sin fine-tuning da")
    print("similitudes uniformemente altas (anisotropía) y discrimina peor.")


In [ ]:
# ============================================================
# Mini buscador semántico: query -> top-5 documentos del corpus
# ============================================================
def semantic_search(query, doc_vecs, sentences, domains, k=5):
    """Rankea los documentos por coseno con la query."""
    if HF_AVAILABLE:
        q = encode_sentences_hf([query])["mean"][0]  # (d,)
    else:
        q = encode_sentences_static([query])[0]
    dv = doc_vecs / (np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-12)
    qn = q / (np.linalg.norm(q) + 1e-12)
    scores = dv @ qn  # (N,)
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), domains[i], sentences[i]) for i in top]

QUERIES = [
    "fallas en el transformador de la subestacion",
    "como entrenar una red neuronal con gradiente",
    "consultas lentas en la base de datos",
    "desplegar funciones en la nube sin servidores",
]

for q in QUERIES:
    print("=" * 90)
    print("QUERY: {}".format(q))
    for score, dom, sent in semantic_search(q, SENT_VECS, SENTENCES, DOMAINS):
        print("  [{:.3f}] ({:<22}) {}".format(score, dom, sent[:70]))
print("=" * 90)
print()
print("Sin compartir NI una palabra exacta, el buscador recupera documentos del tema")
print("correcto: la búsqueda es por significado, no por keywords. (Esto es la base de RAG.)")


<a id="s11"></a>
## 11. Clustering semántico

Si los sentence embeddings capturan el tema, un algoritmo de clustering **sin etiquetas** debería redescubrir los dominios del corpus. Usamos:

- **KMeans** con $k = 5$ (los 5 grupos macro: ML/IA, Eléctrica, Matemáticas, Software, Datos/Infra). KMeans minimiza $\sum_i \|\mathbf{x}_i - \boldsymbol{\mu}_{c(i)}\|^2$ — distancia **euclidiana**, por eso normalizamos los vectores antes (sección 6: con norma 1, euclidiana ≡ coseno).
- **Silhouette score** $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$, donde $a(i)$ es la distancia media al propio cluster y $b(i)$ al cluster ajeno más cercano. Mide cohesión vs separación sin usar etiquetas.
- **Adjusted Rand Index (ARI)** contra los grupos macro reales: 0 ≈ azar, 1 = coincidencia perfecta. Este sí usa las etiquetas, solo para evaluar.


In [ ]:
# ============================================================
# KMeans + silhouette sobre los sentence embeddings
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import normalize

X_sent = normalize(SENT_VECS)  # (N, d) norma 1: euclidiana equivale a coseno
K_CLUSTERS = len(set(GROUPS))  # 5 grupos macro

km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10)
cluster_ids = km.fit_predict(X_sent)  # (N,)

sil = silhouette_score(X_sent, cluster_ids)
ari = adjusted_rand_score(GROUPS, cluster_ids)

print("KMeans con k = {}".format(K_CLUSTERS))
print("Silhouette score: {:.3f}  (interno, sin etiquetas; >0 = clusters con estructura)".format(sil))
print("ARI vs grupos macro: {:.3f}  (0 = azar, 1 = coincidencia perfecta)".format(ari))
print()

# Probar varios k: ¿el k 'natural' coincide con los 5 grupos?
ks = range(2, 10)
sils = []
for k in ks:
    labels_k = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(X_sent)
    sils.append(silhouette_score(X_sent, labels_k))

plt.figure(figsize=(7.5, 4))
plt.plot(list(ks), sils, "o-", lw=2)
plt.axvline(K_CLUSTERS, ls="--", color="red", label="k = {} (grupos macro reales)".format(K_CLUSTERS))
plt.xlabel("número de clusters k"); plt.ylabel("silhouette")
plt.title("Selección de k por silhouette")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# ¿Los clusters coinciden con los dominios? Tabla de contingencia
# ============================================================
df_ct = pd.crosstab(
    pd.Series(GROUPS, name="grupo macro real"),
    pd.Series(["C" + str(c) for c in cluster_ids], name="cluster KMeans"),
)
print("Tabla de contingencia (filas = dominios reales, columnas = clusters):")
display(df_ct)

plt.figure(figsize=(8, 4.5))
sns.heatmap(df_ct, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={"label": "oraciones"})
plt.title("Clusters no supervisados vs grupos de dominio reales\n(ARI = {:.2f})".format(ari))
plt.tight_layout(); plt.show()

# Oraciones representativas: la más cercana al centroide de cada cluster
print("Oración más representativa de cada cluster (la más cercana a su centroide):")
for c in range(K_CLUSTERS):
    members = np.where(cluster_ids == c)[0]
    d2 = np.linalg.norm(X_sent[members] - km.cluster_centers_[c], axis=1)
    rep = members[np.argmin(d2)]
    print("  C{} ({} oraciones): [{}] {}".format(c, len(members), DOMAINS[rep], SENTENCES[rep][:65]))
print()
print("Si una fila se concentra en una columna, el embedding 'sabe' de qué dominio es la")
print("oración sin que nadie se lo haya dicho: estructura temática emergente.")


<a id="s12"></a>
## 12. Visualización de espacios: PCA, t-SNE y UMAP

Nuestros embeddings viven en $d = 64$–$768$ dimensiones; para verlos hay que proyectarlos a 2D/3D. Tres técnicas con filosofías distintas.

### 12.1 PCA — proyección lineal de máxima varianza

**Derivación breve.** Buscamos la dirección $\mathbf{u}$ (con $\|\mathbf{u}\| = 1$) que maximiza la varianza de los datos proyectados. Con datos centrados $X \in \mathbb{R}^{N \times d}$ y matriz de covarianza $C = \frac{1}{N} X^\top X$:

$$\max_{\|\mathbf{u}\|=1} \ \text{Var}(X\mathbf{u}) = \mathbf{u}^\top C\, \mathbf{u}$$

Lagrangiano: $\mathcal{L} = \mathbf{u}^\top C \mathbf{u} - \lambda(\mathbf{u}^\top\mathbf{u} - 1)$. Derivando e igualando a cero:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{u}} = 2C\mathbf{u} - 2\lambda \mathbf{u} = 0 \;\;\Rightarrow\;\; C\mathbf{u} = \lambda\mathbf{u}$$

**La dirección óptima es un eigenvector de la covarianza**, y la varianza capturada es su eigenvalor: $\mathbf{u}^\top C \mathbf{u} = \lambda$. Las componentes principales son los eigenvectores ordenados por eigenvalor; la **varianza explicada** por la componente $j$ es $\lambda_j / \sum_k \lambda_k$ (el *scree plot* la grafica).

Propiedades: lineal (preserva estructura global, distancias grandes son significativas), determinista, rápido. Limitación: si la estructura es no lineal (clusters curvados), 2 componentes pueden capturar poca varianza y mezclar clusters.


In [ ]:
# ============================================================
# PCA: verificación contra eigendecomposición manual + scree + 2D
# ============================================================
from sklearn.decomposition import PCA

# --- Verificación: sklearn PCA == eigenvectores de la covarianza ---
Xc = X_sent - X_sent.mean(axis=0)          # centrar, (N, d)
C = (Xc.T @ Xc) / len(Xc)                  # covarianza, (d, d)
eigvals = np.linalg.eigvalsh(C)[::-1]      # eigenvalores descendentes

pca_full = PCA(n_components=min(30, X_sent.shape[0] - 1, X_sent.shape[1])).fit(Xc)
assert np.allclose(pca_full.explained_variance_[:5], eigvals[:5] * len(Xc) / (len(Xc) - 1), rtol=1e-3)
print("Varianza explicada de sklearn == eigenvalores de la covarianza ✓")

# --- Scree plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
evr = pca_full.explained_variance_ratio_
axes[0].bar(range(1, len(evr) + 1), evr, color="steelblue")
axes[0].plot(range(1, len(evr) + 1), np.cumsum(evr), "o-", color="darkred", ms=3, label="acumulada")
axes[0].set_xlabel("componente principal"); axes[0].set_ylabel("varianza explicada")
axes[0].set_title("Scree plot: PC1+PC2 capturan {:.0%} de la varianza".format(evr[:2].sum()))
axes[0].legend()

# --- Proyección 2D de oraciones, coloreada por grupo macro ---
P2 = pca_full.transform(Xc)[:, :2]  # (N, 2)
group_names = sorted(set(GROUPS))
palette = dict(zip(group_names, sns.color_palette("tab10", len(group_names))))
for g in group_names:
    m = np.array([gg == g for gg in GROUPS])
    axes[1].scatter(P2[m, 0], P2[m, 1], s=28, alpha=0.8, label=g, color=palette[g])
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("PCA 2D de los sentence embeddings")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Si PC1+PC2 explican poca varianza, el mapa 2D es una sombra MUY parcial del espacio:")
print("clusters que se solapan aquí pueden estar bien separados en las demás dimensiones.")


### 12.2 t-SNE — preservar vecindades, no distancias

t-SNE (van der Maaten & Hinton, 2008) no busca varianza: busca que **los vecinos sigan siendo vecinos**. Define similitudes como probabilidades:

- En alta dimensión: $p_{j|i} \propto \exp(-\|\mathbf{x}_i - \mathbf{x}_j\|^2 / 2\sigma_i^2)$ — gaussiana centrada en cada punto.
- En el mapa 2D: $q_{ij} \propto (1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}$ — una **t de Student** de colas pesadas, que da "espacio extra" en 2D para separar puntos medianamente lejanos (alivia el *crowding problem*).

Y minimiza la divergencia KL entre ambas distribuciones:

$$\text{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Intuición de la asimetría de la KL:** el costo es alto cuando $p_{ij}$ es grande y $q_{ij}$ pequeño (separar vecinos reales = muy penalizado), pero barato cuando $p_{ij}$ es pequeño y $q_{ij}$ grande (juntar extraños = poco penalizado). Por eso t-SNE **preserva estructura local y distorsiona la global**.

**Perplexity** controla el $\sigma_i$ de cada punto: es, aproximadamente, el **número efectivo de vecinos** considerados ($\text{Perp}(P_i) = 2^{H(P_i)}$). Baja (~5): estructura muy local, clusters fragmentados. Alta (~50): más global, clusters fusionados. Probaremos tres valores — la lección es que **el mapa cambia con el hiperparámetro**, así que nunca saques conclusiones de una sola corrida.


In [ ]:
# ============================================================
# t-SNE con 3 valores de perplexity
# ============================================================
from sklearn.manifold import TSNE

perplexities = [5, 15, 30]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, perp in zip(axes, perplexities):
    Y = TSNE(n_components=2, perplexity=perp, random_state=SEED, init="pca",
             learning_rate="auto").fit_transform(X_sent)  # (N, 2)
    for g in group_names:
        m = np.array([gg == g for gg in GROUPS])
        ax.scatter(Y[m, 0], Y[m, 1], s=24, alpha=0.8, label=g, color=palette[g])
    ax.set_title("t-SNE, perplexity = {}".format(perp))
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best")
plt.suptitle("El mismo espacio, tres mapas distintos: la perplexity importa", y=1.03)
plt.tight_layout(); plt.show()

print("perplexity baja  -> vecindarios diminutos: clusters fragmentados")
print("perplexity alta  -> vecindarios amplios: estructura más global, menos detalle")
print("Ninguno es 'el verdadero': son lentes distintas sobre el mismo espacio.")


### 12.3 UMAP — la vía topológica

UMAP (McInnes et al., 2018) llega a un objetivo parecido por un camino distinto: modela los datos como un **grafo difuso de vecindades** (una aproximación al *manifold* donde viven los datos, con conexiones ponderadas a los `n_neighbors` vecinos más cercanos), construye el grafo equivalente en 2D y minimiza la entropía cruzada entre ambos grafos.

**vs t-SNE en la práctica:**

| | t-SNE | UMAP |
|---|---|---|
| Preserva | local | local + algo más de **global** |
| Velocidad | lenta ($O(N^2)$ naive, $O(N\log N)$ con Barnes-Hut) | más rápida, escala mejor |
| Nuevos puntos | reentrenar | tiene `transform()` |
| Hiperparámetro clave | `perplexity` | `n_neighbors` (análogo) y `min_dist` |

Las mismas advertencias de interpretación aplican a ambos (ver 12.5).


In [ ]:
# ============================================================
# UMAP (try/except) y comparación lado a lado con PCA y t-SNE
# ============================================================
UMAP_OK = False
try:
    import umap

    reducer = umap.UMAP(n_components=2, n_neighbors=12, min_dist=0.1, random_state=SEED)
    Y_umap = reducer.fit_transform(X_sent)  # (N, 2)
    UMAP_OK = True
except Exception as e:
    print("UMAP no disponible ({}). Instala con: pip install umap-learn".format(type(e).__name__))

if UMAP_OK:
    Y_tsne = TSNE(n_components=2, perplexity=15, random_state=SEED, init="pca",
                  learning_rate="auto").fit_transform(X_sent)
    proyecciones = [("PCA (lineal)", P2), ("t-SNE (perp=15)", Y_tsne), ("UMAP (nn=12)", Y_umap)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for ax, (name, Y) in zip(axes, proyecciones):
        for g in group_names:
            m = np.array([gg == g for gg in GROUPS])
            ax.scatter(Y[m, 0], Y[m, 1], s=24, alpha=0.8, label=g, color=palette[g])
        ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
    axes[0].legend(fontsize=7)
    plt.suptitle("Tres filosofías de proyección sobre los mismos embeddings", y=1.03)
    plt.tight_layout(); plt.show()

    print("PCA conserva geometría global (pero puede solapar clusters);")
    print("t-SNE y UMAP separan vecindarios (pero distorsionan distancias entre clusters).")


In [ ]:
# ============================================================
# Proyección 3D interactiva con plotly (coloreada por grupo de dominio)
# ============================================================
try:
    import plotly.express as px

    P3 = PCA(n_components=3).fit_transform(Xc)  # (N, 3)
    df_3d = pd.DataFrame({
        "PC1": P3[:, 0], "PC2": P3[:, 1], "PC3": P3[:, 2],
        "grupo": GROUPS, "dominio": DOMAINS,
        "oracion": [s[:80] for s in SENTENCES],
    })
    fig = px.scatter_3d(
        df_3d, x="PC1", y="PC2", z="PC3", color="grupo",
        hover_data={"dominio": True, "oracion": True, "PC1": False, "PC2": False, "PC3": False},
        title="Sentence embeddings en 3D (PCA) — pasa el mouse para leer cada oración",
        opacity=0.85,
    )
    fig.update_traces(marker_size=4)
    fig.update_layout(height=600, legend_title_text="grupo macro")
    fig.show()
    print("Rota, hace zoom y pasa el mouse: cada punto es una oración del corpus.")
except Exception as e:
    print("plotly no disponible ({}). Instala con: pip install plotly".format(type(e).__name__))


### 12.5 Advertencias: cómo NO leer un mapa no lineal

Errores comunes al interpretar t-SNE/UMAP (Wattenberg et al., *"How to Use t-SNE Effectively"*, Distill 2016):

1. **El tamaño de los clusters no significa nada.** t-SNE expande regiones densas y comprime las dispersas; un cluster "grande" no es más variado.
2. **Las distancias entre clusters no son fiables.** Que el cluster eléctrico quede "lejos" del de bases de datos no cuantifica su disimilitud real — la KL apenas penaliza colocar mal a los no-vecinos.
3. **Pueden aparecer clusters en ruido puro.** Con perplexity baja, t-SNE fragmenta hasta datos uniformes en "clusters" convincentes.
4. **Cada corrida puede verse distinta** (objetivo no convexo + inicialización aleatoria). Fija seeds y corre varias veces antes de creer una estructura.
5. **Regla práctica:** usa los mapas no lineales para **generar hipótesis** ("parece que hay 5 grupos") y verifícalas con números en el espacio original (silhouette, ARI, vecinos por coseno — secciones 6 y 11). PCA, por ser lineal, sí permite leer distancias globales... pero solo captura la varianza que cabe en 2-3 ejes.


<a id="s13"></a>
## 13. Resumen

El arco completo del notebook:

| Concepto | Idea central | Sección |
|---|---|---|
| One-hot | Sin semántica: todo par es ortogonal | 1 |
| Hipótesis distribucional | El contexto define el significado | 1 |
| PPMI | Co-ocurrencia más allá del azar: $\max(0, \log\frac{P(w,c)}{P(w)P(c)})$ | 1 |
| Skip-gram + NS | Clasificación binaria real/ruido; gradiente $O(Kd)$; negativos $\propto f^{0.75}$ | 2 |
| CBOW | Contexto promediado predice la central; rápido, peor con palabras raras | 3 |
| GloVe | Mínimos cuadrados ponderados sobre $\log X_{ij}$; $f(x)$ saturada | 4 |
| Levy & Goldberg | SGNS factoriza implícitamente $\text{PMI} - \log K$: conteo ≡ predicción | 4 |
| FastText | $\mathbf{v}_w = \sum_g \mathbf{z}_g$: morfología y OOV resueltos por subpalabras | 5 |
| Coseno | Similitud por dirección, inmune a la norma (que crece con la frecuencia) | 6 |
| 3CosAdd | Analogías como traslaciones; frágiles fuera de relaciones paralelas | 7 |
| Contextuales | Una nube por palabra (un punto por ocurrencia); la polisemia se separa por capas | 8 |
| En el transformer | $W_{emb}$ = lookup; tied embeddings; posicional aprendida con estructura de distancia | 9 |
| Sentence embeddings | Mean pooling (con máscara) > CLS sin fine-tuning; base de la búsqueda semántica | 10 |
| Clustering | KMeans + silhouette + ARI: los embeddings recuperan los dominios | 11 |
| PCA / t-SNE / UMAP | Lineal-global vs no-lineal-local; los mapas generan hipótesis, no conclusiones | 12 |

**Conexión con el resto del lab:** en el notebook 05 construiremos el transformer cuya primera capa es exactamente la $W_{emb}$ de la sección 9 — pero entrenada end-to-end con next-token prediction, produciendo los embeddings contextuales de la sección 8 desde dentro.


## Ejercicios propuestos

1. **(Fácil)** Modifica `MIN_COUNT` a 1 y reentrena skip-gram. ¿Qué pasa con los vecinos de las palabras que aparecían una sola vez? Relaciónalo con la discusión skip-gram vs CBOW de la sección 3.

2. **(Fácil)** En la sección 6, calcula los vecinos de "de" y "la" con producto punto y con coseno. ¿Cuál medida las "cuela" entre los vecinos de palabras de contenido? Explica con la gráfica norma-vs-frecuencia.

3. **(Media)** Implementa **3CosMul** (Levy & Goldberg, 2014): $b^* = \arg\max_x \frac{\cos(x, b)\,\cos(x, a^*)}{\cos(x, a) + \epsilon}$ y compara con 3CosAdd en las analogías de la sección 7. ¿Cuál es más robusto cuando un término de la consulta domina?

4. **(Media)** Entrena skip-gram con `window=1` y con `window=8` (ajusta epochs). Compara los vecinos de "transformador": ¿la ventana corta da vecinos más sintácticos/posicionales y la larga más temáticos? (Es el resultado clásico de Levy & Goldberg, 2014b sobre ventanas.)

5. **(Difícil)** Implementa la factorización **SVD truncada de la matriz PPMI** (`np.linalg.svd` o `TruncatedSVD`) con $d = $ `CFG["emb_dim"]` y compara sus vecinos y analogías contra skip-gram y GloVe. Estás reproduciendo el experimento central de Levy, Goldberg & Dagan (2015): con los hiperparámetros correctos, ¿el método de conteo alcanza al predictivo?


## Referencias

**Papers fundacionales:**

- Mikolov, T. et al. (2013a). *Efficient Estimation of Word Representations in Vector Space.* — skip-gram y CBOW. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- Mikolov, T. et al. (2013b). *Distributed Representations of Words and Phrases and their Compositionality.* — negative sampling, subsampling, $f^{3/4}$. [arXiv:1310.4546](https://arxiv.org/abs/1310.4546)
- Pennington, J., Socher, R. & Manning, C. (2014). *GloVe: Global Vectors for Word Representation.* [Paper + vectores](https://nlp.stanford.edu/projects/glove/) · [PDF (EMNLP)](https://aclanthology.org/D14-1162/)
- Bojanowski, P. et al. (2017). *Enriching Word Vectors with Subword Information.* — FastText. [arXiv:1607.04606](https://arxiv.org/abs/1607.04606)
- Levy, O. & Goldberg, Y. (2014). *Neural Word Embedding as Implicit Matrix Factorization.* — SGNS ≈ PMI − log k. [NeurIPS 2014](https://papers.nips.cc/paper/2014/hash/feab05aa91085b7a8012516bc3533958-Abstract.html)
- Levy, O. & Goldberg, Y. (2014b). *Linguistic Regularities in Sparse and Explicit Word Representations.* — 3CosAdd/3CosMul. [CoNLL 2014](https://aclanthology.org/W14-1618/)

**Contextuales y análisis:**

- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers.* [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- Ethayarajh, K. (2019). *How Contextual are Contextualized Word Representations?* — anisotropía por capas. [arXiv:1909.00512](https://arxiv.org/abs/1909.00512)
- Press, O. & Wolf, L. (2017). *Using the Output Embedding to Improve Language Models.* — weight tying. [arXiv:1608.05859](https://arxiv.org/abs/1608.05859)
- Bolukbasi, T. et al. (2016). *Man is to Computer Programmer as Woman is to Homemaker?* — sesgo en embeddings. [arXiv:1607.06520](https://arxiv.org/abs/1607.06520)

**Visualización:**

- van der Maaten, L. & Hinton, G. (2008). *Visualizing Data using t-SNE.* [JMLR](https://www.jmlr.org/papers/v9/vandermaaten08a.html)
- McInnes, L., Healy, J. & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection.* [arXiv:1802.03426](https://arxiv.org/abs/1802.03426)
- Wattenberg, M., Viégas, F. & Johnson, I. (2016). *How to Use t-SNE Effectively.* [Distill](https://distill.pub/2016/misread-tsne/)

**Material de curso:**

- Jurafsky, D. & Martin, J. *Speech and Language Processing* (3a ed.), cap. 6 — Vector Semantics and Embeddings. [Web](https://web.stanford.edu/~jurafsky/slp3/)
- Stanford CS224N, Lectures 1-2 — Word Vectors. [Curso](https://web.stanford.edu/class/cs224n/)
